<a href="https://colab.research.google.com/github/Misha-private/Demo-repo/blob/main/Emulator_AugExtend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 ## Connect to /content/drive

In [1]:
from google.colab import drive
import os
def is_drive_mounted():
    return os.path.exists('/content/drive')
if not is_drive_mounted(): drive.mount('/content/drive')


Mounted at /content/drive


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Flush and unmount

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


# Script for building extneded ICs

In [ ]:
# ============================================================
# make_ic_bank_big50plus_ai_emul3.py
# ------------------------------------------------------------
# Generate expanded IC bank for 1-layer SWE Klein beta-plane.
#
# New IC families:
#   kelvin_00 ... kelvin_15
#   rh_kelvin_00 ... rh_kelvin_15
#   modon_kelvin_00 ... modon_kelvin_15
#   wavepacket_00 ... wavepacket_15
#   rh_phaseamp_00 ... rh_phaseamp_15
#
# Saves each IC as:
#   /content/drive/MyDrive/AI_EMUL3/ic_bank_big50plus/<IC_KEY>/ic.npz
#   /content/drive/MyDrive/AI_EMUL3/ic_bank_big50plus/<IC_KEY>/preview.png
#
# The file contains:
#   eta, uc, vc, x, y, f, H, G, Lx, Ly, nx, ny, ic_key, family
#
# IMPORTANT:
#   This script only creates initial conditions.
#   Next step: run your FD solver from these ICs and save trajectories to:
#       /content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50plus/<IC_KEY>/
# ============================================================

import os
import json
import math
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
class CFG:
    OUT_ROOT = "/content/drive/MyDrive/AI_EMUL3/ic_bank_big50plus"

    nx = 256
    ny = 128
    Lx = 2.0e7
    Ly = 8.0e6

    H = 1000.0
    G = 9.81

    # Simple beta/Klein-like Coriolis profile on centers.
    # Adjust fp if your FD model uses a different value.
    fp = 1.0e-4

    N_PER_FAMILY = 16
    SEED = 1234

    # Amplitude ranges
    ETA_AMP_KELVIN = (6.0, 25.0)      # meters
    ETA_AMP_RH     = (15.0, 55.0)     # meters
    ETA_AMP_MODON  = (20.0, 70.0)     # meters
    ETA_AMP_PACKET = (8.0, 35.0)      # meters

    # Velocity clipping for safe ICs
    U_CLIP = 80.0

cfg = CFG()
os.makedirs(cfg.OUT_ROOT, exist_ok=True)

rng = np.random.default_rng(cfg.SEED)

# ------------------------------------------------------------
# Grid
# ------------------------------------------------------------
x = np.linspace(0.0, cfg.Lx, cfg.nx, endpoint=False)
y = np.linspace(-0.5 * cfg.Ly, 0.5 * cfg.Ly, cfg.ny, endpoint=False)
X, Y = np.meshgrid(x, y)

x_norm = 2.0 * np.pi * X / cfg.Lx
y_norm = np.pi * Y / (0.5 * cfg.Ly)

# Coriolis-like profile
f = cfg.fp * np.sin(np.pi * Y / cfg.Ly)

dx = cfg.Lx / cfg.nx
dy = cfg.Ly / cfg.ny

# ------------------------------------------------------------
# Basic numerical helpers
# ------------------------------------------------------------
def ddx_periodic(q):
    return (np.roll(q, -1, axis=1) - np.roll(q, 1, axis=1)) / (2.0 * dx)

def ddy_edge(q):
    qp = np.empty_like(q)
    qm = np.empty_like(q)
    qp[:-1, :] = q[1:, :]
    qp[-1, :]  = q[-1, :]
    qm[1:, :]  = q[:-1, :]
    qm[0, :]   = q[0, :]
    return (qp - qm) / (2.0 * dy)

def smooth_window_y(width_frac=0.12):
    yy = (Y - y.min()) / (y.max() - y.min())
    w1 = 0.5 * (1.0 + np.tanh((yy - width_frac) / (0.35 * width_frac)))
    w2 = 0.5 * (1.0 + np.tanh(((1.0 - width_frac) - yy) / (0.35 * width_frac)))
    return w1 * w2

WY = smooth_window_y()

def geostrophic_from_eta(eta, fmin=2.0e-5):
    eta_x = ddx_periodic(eta)
    eta_y = ddy_edge(eta)

    f_safe = np.sign(f) * np.maximum(np.abs(f), fmin)
    f_safe[f_safe == 0] = fmin

    uc = -(cfg.G / f_safe) * eta_y
    vc =  (cfg.G / f_safe) * eta_x

    uc = np.clip(uc, -cfg.U_CLIP, cfg.U_CLIP)
    vc = np.clip(vc, -cfg.U_CLIP, cfg.U_CLIP)
    return uc, vc

def add_kelvin_balance(eta, phase_speed=None, direction=1.0, meridional_sign=1.0):
    if phase_speed is None:
        phase_speed = math.sqrt(cfg.G * cfg.H)

    uc = direction * phase_speed * eta / cfg.H

    # Weak cross-stream component, mostly to avoid exactly zero v.
    vc = 0.08 * meridional_sign * phase_speed * (eta / cfg.H) * np.sin(np.pi * Y / cfg.Ly)

    uc = np.clip(uc, -cfg.U_CLIP, cfg.U_CLIP)
    vc = np.clip(vc, -cfg.U_CLIP, cfg.U_CLIP)
    return uc, vc

def combine_states(parts):
    eta = np.zeros((cfg.ny, cfg.nx), dtype=np.float64)
    uc  = np.zeros_like(eta)
    vc  = np.zeros_like(eta)

    for e, u, v, weight in parts:
        eta += weight * e
        uc  += weight * u
        vc  += weight * v

    eta = eta - eta.mean()
    uc = np.clip(uc, -cfg.U_CLIP, cfg.U_CLIP)
    vc = np.clip(vc, -cfg.U_CLIP, cfg.U_CLIP)
    return eta.astype(np.float32), uc.astype(np.float32), vc.astype(np.float32)

# ------------------------------------------------------------
# IC families
# ------------------------------------------------------------
def make_kelvin(seed_i):
    amp = rng.uniform(*cfg.ETA_AMP_KELVIN)
    k = rng.integers(1, 5)
    phase = rng.uniform(0, 2*np.pi)

    y0 = rng.uniform(-0.25 * cfg.Ly, 0.25 * cfg.Ly)
    mer_width = rng.uniform(0.10, 0.28) * cfg.Ly

    envelope = np.exp(-((Y - y0) / mer_width) ** 2)
    wave = np.cos(k * x_norm + phase)

    eta = amp * envelope * wave * WY
    eta -= eta.mean()

    direction = rng.choice([-1.0, 1.0])
    mer_sign = rng.choice([-1.0, 1.0])
    uc, vc = add_kelvin_balance(eta, direction=direction, meridional_sign=mer_sign)
    return eta, uc, vc

def make_rh_like(seed_i):
    amp = rng.uniform(*cfg.ETA_AMP_RH)
    m = rng.integers(2, 6)
    n = rng.integers(1, 4)
    phase = rng.uniform(0, 2*np.pi)

    mer = np.cos(n * np.pi * Y / cfg.Ly)
    mer2 = 0.35 * np.cos((n + 1) * np.pi * Y / cfg.Ly + rng.uniform(0, 2*np.pi))
    zonal = np.cos(m * x_norm + phase)
    tilt = 0.3 * np.sin((m + 1) * x_norm - 0.7 * y_norm + phase)

    eta = amp * (mer + mer2) * (zonal + tilt) * WY
    eta -= eta.mean()

    uc, vc = geostrophic_from_eta(eta)
    return eta, uc, vc

def make_modon_like(seed_i):
    amp = rng.uniform(*cfg.ETA_AMP_MODON)
    x0 = rng.uniform(0.15 * cfg.Lx, 0.85 * cfg.Lx)
    y0 = rng.uniform(-0.25 * cfg.Ly, 0.25 * cfg.Ly)

    sx = rng.uniform(0.07, 0.16) * cfg.Lx
    sy = rng.uniform(0.10, 0.22) * cfg.Ly
    sep = rng.uniform(0.08, 0.18) * cfg.Lx

    def pdist_x(X, x0):
        d = X - x0
        d = (d + 0.5 * cfg.Lx) % cfg.Lx - 0.5 * cfg.Lx
        return d

    r1 = (pdist_x(X, x0 - 0.5 * sep) / sx) ** 2 + ((Y - y0) / sy) ** 2
    r2 = (pdist_x(X, x0 + 0.5 * sep) / sx) ** 2 + ((Y - y0) / sy) ** 2

    eta = amp * (np.exp(-r1) - np.exp(-r2)) * WY
    eta -= eta.mean()

    uc, vc = geostrophic_from_eta(eta)
    return eta, uc, vc

def make_wavepacket(seed_i):
    amp = rng.uniform(*cfg.ETA_AMP_PACKET)

    nmodes = rng.integers(3, 7)
    eta = np.zeros((cfg.ny, cfg.nx), dtype=np.float64)

    x0 = rng.uniform(0, cfg.Lx)
    y0 = rng.uniform(-0.25 * cfg.Ly, 0.25 * cfg.Ly)
    sx = rng.uniform(0.15, 0.35) * cfg.Lx
    sy = rng.uniform(0.20, 0.45) * cfg.Ly

    dxp = (X - x0 + 0.5 * cfg.Lx) % cfg.Lx - 0.5 * cfg.Lx
    envelope = np.exp(-(dxp/sx)**2 - ((Y-y0)/sy)**2)

    for _ in range(nmodes):
        kx = rng.integers(1, 7)
        ky = rng.integers(1, 5)
        phase = rng.uniform(0, 2*np.pi)
        coeff = rng.normal()
        eta += coeff * np.cos(kx*x_norm + ky*y_norm + phase)

    eta = amp * eta / (np.std(eta) + 1e-8)
    eta = eta * envelope * WY
    eta -= eta.mean()

    ug, vg = geostrophic_from_eta(eta)
    uk, vk = add_kelvin_balance(eta, direction=rng.choice([-1.0, 1.0]))
    alpha = rng.uniform(0.25, 0.55)
    uc = (1-alpha)*ug + alpha*uk
    vc = (1-alpha)*vg + alpha*vk
    uc = np.clip(uc, -cfg.U_CLIP, cfg.U_CLIP)
    vc = np.clip(vc, -cfg.U_CLIP, cfg.U_CLIP)
    return eta, uc, vc

# ------------------------------------------------------------
# Save/plot
# ------------------------------------------------------------
def save_ic(ic_key, family, eta, uc, vc, meta):
    out_dir = os.path.join(cfg.OUT_ROOT, ic_key)
    os.makedirs(out_dir, exist_ok=True)

    np.savez_compressed(
        os.path.join(out_dir, "ic.npz"),
        eta=eta.astype(np.float32),
        uc=uc.astype(np.float32),
        vc=vc.astype(np.float32),
        x=x.astype(np.float32),
        y=y.astype(np.float32),
        f=f.astype(np.float32),
        H=np.float32(cfg.H),
        G=np.float32(cfg.G),
        Lx=np.float32(cfg.Lx),
        Ly=np.float32(cfg.Ly),
        nx=np.int32(cfg.nx),
        ny=np.int32(cfg.ny),
        ic_key=ic_key,
        family=family,
    )

    with open(os.path.join(out_dir, "meta.json"), "w") as fp:
        json.dump(meta, fp, indent=2)

    fig, axs = plt.subplots(1, 3, figsize=(13, 3.5))
    for ax, fld, title in zip(axs, [eta, uc, vc], ["eta", "uc", "vc"]):
        vmax = np.nanmax(np.abs(fld))
        if vmax == 0:
            vmax = 1.0
        im = ax.imshow(fld, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        ax.set_title(f"{ic_key}: {title}")
        ax.set_xticks([])
        ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "preview.png"), dpi=150, bbox_inches="tight")
    plt.close()

def summarize_field(name, eta, uc, vc):
    ke = np.mean(0.5 * (cfg.H + eta) * (uc**2 + vc**2))
    return {
        "ic_key": name,
        "eta_min": float(eta.min()),
        "eta_max": float(eta.max()),
        "eta_std": float(eta.std()),
        "u_min": float(uc.min()),
        "u_max": float(uc.max()),
        "u_std": float(uc.std()),
        "v_min": float(vc.min()),
        "v_max": float(vc.max()),
        "v_std": float(vc.std()),
        "ke_mean": float(ke),
    }

def main():
    print("[IC Bank] output:", cfg.OUT_ROOT)
    summaries = []

    # 1. Kelvin
    for i in range(cfg.N_PER_FAMILY):
        ic_key = f"kelvin_{i:02d}"
        eta, uc, vc = make_kelvin(i)
        save_ic(ic_key, "kelvin", eta, uc, vc, {"family": "kelvin", "index": i})
        summaries.append(summarize_field(ic_key, eta, uc, vc))

    # 2. RH + Kelvin
    for i in range(cfg.N_PER_FAMILY):
        ic_key = f"rh_kelvin_{i:02d}"
        e1, u1, v1 = make_rh_like(i)
        e2, u2, v2 = make_kelvin(i)
        alpha = rng.uniform(0.25, 0.55)
        eta, uc, vc = combine_states([(e1, u1, v1, 1.0), (e2, u2, v2, alpha)])
        save_ic(ic_key, "rh_kelvin", eta, uc, vc, {"family": "rh_kelvin", "index": i, "kelvin_weight": float(alpha)})
        summaries.append(summarize_field(ic_key, eta, uc, vc))

    # 3. Modon + Kelvin
    for i in range(cfg.N_PER_FAMILY):
        ic_key = f"modon_kelvin_{i:02d}"
        e1, u1, v1 = make_modon_like(i)
        e2, u2, v2 = make_kelvin(i)
        alpha = rng.uniform(0.20, 0.50)
        eta, uc, vc = combine_states([(e1, u1, v1, 1.0), (e2, u2, v2, alpha)])
        save_ic(ic_key, "modon_kelvin", eta, uc, vc, {"family": "modon_kelvin", "index": i, "kelvin_weight": float(alpha)})
        summaries.append(summarize_field(ic_key, eta, uc, vc))

    # 4. Wavepacket
    for i in range(cfg.N_PER_FAMILY):
        ic_key = f"wavepacket_{i:02d}"
        eta, uc, vc = make_wavepacket(i)
        save_ic(ic_key, "wavepacket", eta, uc, vc, {"family": "wavepacket", "index": i})
        summaries.append(summarize_field(ic_key, eta, uc, vc))

    # 5. RH phase/amplitude variants
    for i in range(cfg.N_PER_FAMILY):
        ic_key = f"rh_phaseamp_{i:02d}"
        eta, uc, vc = make_rh_like(i)
        amp_scale = rng.uniform(0.65, 1.35)
        eta = amp_scale * eta
        uc = amp_scale * uc
        vc = amp_scale * vc
        eta -= eta.mean()
        uc = np.clip(uc, -cfg.U_CLIP, cfg.U_CLIP)
        vc = np.clip(vc, -cfg.U_CLIP, cfg.U_CLIP)
        save_ic(ic_key, "rh_phaseamp", eta, uc, vc, {"family": "rh_phaseamp", "index": i, "amp_scale": float(amp_scale)})
        summaries.append(summarize_field(ic_key, eta, uc, vc))

    import pandas as pd
    df = pd.DataFrame(summaries)
    csv_path = os.path.join(cfg.OUT_ROOT, "ic_summary.csv")
    df.to_csv(csv_path, index=False)

    print("[DONE] generated ICs:", len(df))
    print("Summary:", csv_path)
    print(df.head())
    print(df.tail())

if __name__ == "__main__":
    main()


[IC Bank] output: /content/drive/MyDrive/AI_EMUL3/ic_bank_big50plus
[DONE] generated ICs: 80
Summary: /content/drive/MyDrive/AI_EMUL3/ic_bank_big50plus/ic_summary.csv
      ic_key    eta_min    eta_max   eta_std     u_min     u_max     u_std  \
0  kelvin_00 -24.552476  24.552476  7.703388 -2.431811  2.431811  0.762985   
1  kelvin_01 -10.576680  10.576680  3.109710 -1.047572  1.047572  0.308003   
2  kelvin_02 -17.580029  17.580029  6.478378 -1.741222  1.741222  0.641654   
3  kelvin_03 -10.226552  10.226552  2.688936 -1.012893  1.012893  0.266327   
4  kelvin_04 -18.716744  18.716744  5.905437 -1.853808  1.853808  0.584907   

      v_min     v_max     v_std     ke_mean  
0 -0.089786  0.089786  0.025491  291.398289  
1 -0.058728  0.058728  0.016034   47.561346  
2 -0.062889  0.062889  0.020311  206.066097  
3 -0.047244  0.047244  0.011894   35.535721  
4 -0.107500  0.107500  0.030352  171.518483  
            ic_key    eta_min    eta_max    eta_std      u_min      u_max  \
75  rh_phas

# Convert derived ICs in a single bundle

In [ ]:
# ============================================================
# make_bundle_from_ic_bank_big50plus_ai_emul3.py
# ------------------------------------------------------------
# Convert folder-based IC bank:
#
#   /content/drive/MyDrive/AI_EMUL3/ic_bank_big50plus/<IC_KEY>/ic.npz
#
# into one old-style bundle file:
#
#   /content/drive/MyDrive/AI_EMUL3/klein_ics_1L_big50plus/bundle_cgrid_1L_big50plus.npz
#
# This allows the existing FD driver to keep using:
#
#   IC_BUNDLE = ".../bundle_cgrid_1L_big50plus.npz"
#
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
IC_BANK_DIR = "/content/drive/MyDrive/AI_EMUL3/ic_bank_big50plus"

OUT_DIR = "/content/drive/MyDrive/AI_EMUL3/klein_ics_1L_big50plus"
OUT_BUNDLE = f"{OUT_DIR}/bundle_cgrid_1L_big50plus.npz"
OUT_SUMMARY = f"{OUT_DIR}/bundle_summary.csv"

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Build bundle
# ------------------------------------------------------------
ic_dirs = sorted([
    d for d in glob.glob(os.path.join(IC_BANK_DIR, "*"))
    if os.path.isdir(d) and os.path.exists(os.path.join(d, "ic.npz"))
])

if len(ic_dirs) == 0:
    raise RuntimeError(f"No IC folders with ic.npz found in: {IC_BANK_DIR}")

bundle = {}
rows = []

for d in ic_dirs:
    ic_key = os.path.basename(d)
    path = os.path.join(d, "ic.npz")
    z = np.load(path, allow_pickle=True)

    eta = z["eta"].astype(np.float32)
    uc  = z["uc"].astype(np.float32)
    vc  = z["vc"].astype(np.float32)

    family = str(z["family"]) if "family" in z.files else "unknown"

    item = {
        "eta": eta,
        "uc": uc,
        "vc": vc,
    }

    for key in ["x", "y", "f", "H", "G", "Lx", "Ly", "nx", "ny", "ic_key", "family"]:
        if key in z.files:
            item[key] = z[key]

    bundle[ic_key] = item

    H = float(z["H"]) if "H" in z.files else 1000.0
    ke_mean = float(np.mean(0.5 * (H + eta) * (uc**2 + vc**2)))

    rows.append({
        "ic_key": ic_key,
        "family": family,
        "eta_min": float(eta.min()),
        "eta_max": float(eta.max()),
        "eta_std": float(eta.std()),
        "u_min": float(uc.min()),
        "u_max": float(uc.max()),
        "u_std": float(uc.std()),
        "v_min": float(vc.min()),
        "v_max": float(vc.max()),
        "v_std": float(vc.std()),
        "ke_mean": ke_mean,
    })

np.savez_compressed(OUT_BUNDLE, **bundle)

df = pd.DataFrame(rows)
df.to_csv(OUT_SUMMARY, index=False)

print("[DONE]")
print("Number of ICs:", len(bundle))
print("Bundle saved to:", OUT_BUNDLE)
print("Summary saved to:", OUT_SUMMARY)
print()
print(df.head())
print(df.tail())


[DONE]
Number of ICs: 80
Bundle saved to: /content/drive/MyDrive/AI_EMUL3/klein_ics_1L_big50plus/bundle_cgrid_1L_big50plus.npz
Summary saved to: /content/drive/MyDrive/AI_EMUL3/klein_ics_1L_big50plus/bundle_summary.csv

      ic_key  family    eta_min    eta_max   eta_std     u_min     u_max  \
0  kelvin_00  kelvin -24.552477  24.552477  7.703388 -2.431811  2.431811   
1  kelvin_01  kelvin -10.576680  10.576680  3.109710 -1.047572  1.047572   
2  kelvin_02  kelvin -17.580029  17.580029  6.478378 -1.741222  1.741222   
3  kelvin_03  kelvin -10.226552  10.226552  2.688936 -1.012893  1.012893   
4  kelvin_04  kelvin -18.716743  18.716743  5.905437 -1.853808  1.853808   

      u_std     v_min     v_max     v_std     ke_mean  
0  0.762985 -0.089786  0.089786  0.025491  291.398285  
1  0.308003 -0.058728  0.058728  0.016034   47.561344  
2  0.641654 -0.062889  0.062889  0.020311  206.066086  
3  0.266327 -0.047244  0.047244  0.011894   35.535721  
4  0.584907 -0.107500  0.107500  0.030352  

# FD model applied to extended IC

In [ ]:
# ============================================================
# run_fd_1layer_catalog_with_colloc.py
# ------------------------------------------------------------
# Run 1-layer shallow-water FD model on twisted/Klein beta-plane
# for every IC in a C-grid bundle.
#
# INPUT:
#   /content/drive/MyDrive/AI_4DVAR/klein_ics_1L/bundle_cgrid_1L.npz
#
# OUTPUT:
#   /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/<IC_KEY>/klein_step_XXXXXX.npz
#   plus plots and time series
#
# NEW OUTPUT:
#   /content/drive/MyDrive/AI_4DVAR/klein_ml_1L/colloc_raw/<IC_KEY>_colloc.npz
#
# The collocation file stores isolated centered grid points plus local FD
# diagnostics at those points:
#   t_sec, fd_step, macro_index, ix, iy, x_m, y_m
#   eta, uc, vc, f
#   deta_dt_fd, duc_dt_fd, dvc_dt_fd
#   etax_fd, etay_fd
#   ucx_fd, ucy_fd
#   vcx_fd, vcy_fd
#   div_fd, zeta_fd
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Paths
# -------------------------
#IC_BUNDLE = "/content/drive/MyDrive/AI_4DVAR/klein_ics_1L/bundle_cgrid_1L.npz"
#ROOT_OUT  = "/content/drive/MyDrive/AI_EMUL/klein_ckpt_1L_big50"
#ROOT_COLLOC = "/content/drive/MyDrive/AI_EMUL/klein_ml_1L_big50/colloc_raw"

IC_BUNDLE = "/content/drive/MyDrive/AI_EMUL3/klein_ics_1L_big50plus/bundle_cgrid_1L_big50plus.npz"
ROOT_OUT = "/content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50plus"
ROOT_COLLOC = "/content/drive/MyDrive/AI_EMUL3/klein_ml_1L_big50plus/colloc_raw"


# -------------------------
# Physics / grid
# -------------------------
g = 9.81
H = 1000.0
nx = 256
ny = 128
Lx = 2.0e7
Ly = 8.0e6
dx = Lx / nx
dy = Ly / ny
fp = 8.0e-5

dt = 30.0
nt = 1200
SAVE_STEPS = set(range(0, nt + 1, 50))

# -------------------------
# Colab batch controls
# -------------------------
# Run only keys[BATCH_START:BATCH_END] in this Colab session.
# Set BATCH_END = None to run to the end.
BATCH_START = 20
BATCH_END   = 40

# Restart-safe behavior: if final snapshot exists, skip that IC.
SKIP_COMPLETED = True
FINAL_SNAPSHOT = f"klein_step_{nt:06d}.npz"

# diffusion
nu2_u, nu2_v, nu2_h = 1.0e4, 1.0e4, 5.0e3
nu4_u, nu4_v, nu4_h = 5.0e10, 5.0e10, 2.5e10

HMIN = 0.5

# -------------------------
# Collocation config
# -------------------------
SAVE_COLLOC = True
COLLOC_EVERY_N_STEPS = 20
N_COLLOC_PER_TIME = 256
COLLOC_WEIGHT_MODE = "grad_eta"   # "uniform" or "grad_eta"
COLLOC_SEED = 42
SAVE_COLLOC_COORDS = True

# -------------------------
# Geometry
# -------------------------
x_c = np.linspace(0.5 * dx, Lx - 0.5 * dx, nx)
y_c = np.linspace(0.5 * dy, Ly - 0.5 * dy, ny)
Xc, Yc = np.meshgrid(x_c, y_c)

phi_c = np.pi * ((Yc / Ly) - 0.5)
f_c = fp * np.sin(phi_c)

x_u = np.linspace(0.0, Lx, nx + 1)
y_v = np.linspace(0.0, Ly, ny + 1)
Xu, Yu = np.meshgrid(x_u, y_c)
Xv, Yv = np.meshgrid(x_c, y_v)

phi_u = np.pi * ((Yu / Ly) - 0.5)
phi_v = np.pi * ((Yv / Ly) - 0.5)
f_u = fp * np.sin(phi_u)
f_v = fp * np.sin(phi_v)

# -------------------------
# Klein twist BCs
# -------------------------
def twist_reflect_x(arr):
    return arr[..., ::-1]

def apply_bc_1l(u, v, h):
    # centers even
    h[0, :]  = 0.5 * (h[1, :]  + twist_reflect_x(h[1, :]))
    h[-1, :] = 0.5 * (h[-2, :] + twist_reflect_x(h[-2, :]))

    # u odd
    u[0, :]  = 0.5 * (u[1, :]  - twist_reflect_x(u[1, :]))
    u[-1, :] = 0.5 * (u[-2, :] - twist_reflect_x(u[-2, :]))

    # v even
    v[0, :]  = 0.5 * (v[1, :]  + twist_reflect_x(v[1, :]))
    v[-1, :] = 0.5 * (v[-2, :] + twist_reflect_x(v[-2, :]))
    return u, v, h

# -------------------------
# C-grid helpers
# -------------------------
def center_from_u(u):
    return 0.5 * (u[:, :-1] + u[:, 1:])

def center_from_v(v):
    return 0.5 * (v[:-1, :] + v[1:, :])

def avg_x(a):
    return 0.5 * (
        np.pad(a, ((0, 0), (1, 0)), mode="wrap")
        + np.pad(a, ((0, 0), (0, 1)), mode="wrap")
    )

def avg_y(a):
    return 0.5 * (
        np.pad(a, ((1, 0), (0, 0)), mode="edge")
        + np.pad(a, ((0, 1), (0, 0)), mode="edge")
    )

def ddx_c_to_u(phi):
    L = np.pad(phi, ((0, 0), (1, 0)), mode="wrap")
    R = np.pad(phi, ((0, 0), (0, 1)), mode="wrap")
    return (R - L) / (2.0 * dx)

def ddy_c_to_v(phi):
    T = np.pad(phi, ((1, 0), (0, 0)), mode="edge")
    B = np.pad(phi, ((0, 1), (0, 0)), mode="edge")
    return (B - T) / (2.0 * dy)

def ddx_u_to_c(phi_u):
    return (phi_u[:, 1:] - phi_u[:, :-1]) / dx

def ddy_v_to_c(phi_v):
    return (phi_v[1:, :] - phi_v[:-1, :]) / dy

# -------------------------
# Centered FD diagnostics on emulator variables
# Used only for saved collocation diagnostics, not solver evolution.
# -------------------------
def ddx_center(a):
    return (np.roll(a, -1, axis=1) - np.roll(a, 1, axis=1)) / (2.0 * dx)

def ddy_center_edge(a):
    ap = np.pad(a, ((1, 1), (0, 0)), mode="edge")
    return (ap[2:, :] - ap[:-2, :]) / (2.0 * dy)

# -------------------------
# Laplacians
# -------------------------
def lap_u(u):
    ue = np.pad(u, ((0, 0), (1, 1)), mode="wrap")
    u_xx = (ue[:, :-2] - 2 * ue[:, 1:-1] + ue[:, 2:]) / dx**2

    ue2 = np.pad(u, ((1, 1), (0, 0)), mode="edge")
    u_yy = (ue2[:-2, :] - 2 * ue2[1:-1, :] + ue2[2:, :]) / dy**2
    return u_xx + u_yy

def lap_v(v):
    ve = np.pad(v, ((0, 0), (1, 1)), mode="wrap")
    v_xx = (ve[:, :-2] - 2 * ve[:, 1:-1] + ve[:, 2:]) / dx**2

    ve2 = np.pad(v, ((1, 1), (0, 0)), mode="edge")
    v_yy = (ve2[:-2, :] - 2 * ve2[1:-1, :] + ve2[2:, :]) / dy**2
    return v_xx + v_yy

def lap_c(h):
    he = np.pad(h, ((0, 0), (1, 1)), mode="wrap")
    h_xx = (he[:, :-2] - 2 * he[:, 1:-1] + he[:, 2:]) / dx**2

    he2 = np.pad(h, ((1, 1), (0, 0)), mode="edge")
    h_yy = (he2[:-2, :] - 2 * he2[1:-1, :] + he2[2:, :]) / dy**2
    return h_xx + h_yy

def bih_u(u):
    return lap_u(lap_u(u))

def bih_v(v):
    return lap_v(lap_v(v))

def bih_c(h):
    return lap_c(lap_c(h))

# -------------------------
# Vorticity
# -------------------------
def compute_corner_vort(u, v):
    v_w = np.pad(v, ((0, 0), (1, 0)), mode="wrap")
    v_e = np.pad(v, ((0, 0), (0, 1)), mode="wrap")
    dv_dx = (v_e - v_w) / (2 * dx)

    u_s = np.pad(u, ((1, 0), (0, 0)), mode="edge")
    u_n = np.pad(u, ((0, 1), (0, 0)), mode="edge")
    du_dy = (u_n - u_s) / (2 * dy)

    return dv_dx - du_dy

def to_u_from_corners(a):
    return 0.5 * (a[:-1, :] + a[1:, :])

def to_v_from_corners(a):
    return 0.5 * (a[:, :-1] + a[:, 1:])

# -------------------------
# Positivity floor
# -------------------------
def enforce_floor_ke_preserving(u, v, h, hmin=HMIN):
    mask = (h < hmin)
    if not np.any(mask):
        return u, v, h, 0

    s_c = np.ones_like(h, dtype=np.float32)
    s_c[mask] = np.sqrt(np.maximum(h[mask], 1e-12) / hmin)

    s_u = 0.5 * (
        np.pad(s_c, ((0, 0), (1, 0)), mode="wrap")
        + np.pad(s_c, ((0, 0), (0, 1)), mode="wrap")
    )
    s_v = 0.5 * (
        np.pad(s_c, ((1, 0), (0, 0)), mode="edge")
        + np.pad(s_c, ((0, 1), (0, 0)), mode="edge")
    )

    u = u * s_u
    v = v * s_v
    h = np.maximum(h, hmin)
    return u, v, h, int(mask.sum())

# -------------------------
# RHS (1-layer AL-style)
# -------------------------
def rhs_1l(u, v, h):
    u, v, h = apply_bc_1l(u, v, h)

    uc = center_from_u(u)
    vc = center_from_v(v)
    K = 0.5 * (uc**2 + vc**2)

    eta = h - H
    Phi = g * eta

    dPhidx_u = ddx_c_to_u(Phi)
    dPhidy_v = ddy_c_to_v(Phi)
    dKdx_u   = ddx_c_to_u(K)
    dKdy_v   = ddy_c_to_v(K)

    z_corners = compute_corner_vort(u, v)
    eta_u = to_u_from_corners(z_corners) + f_u
    eta_v = to_v_from_corners(z_corners) + f_v

    v_tu = avg_x(center_from_v(v))
    u_tv = avg_y(center_from_u(u))

    du = -(dPhidx_u + dKdx_u) + eta_u * v_tu + nu2_u * lap_u(u) + nu4_u * bih_u(u)
    dv = -(dPhidy_v + dKdy_v) - eta_v * u_tv + nu2_v * lap_v(v) + nu4_v * bih_v(v)

    h_u = avg_x(h)
    h_v = avg_y(h)
    F_u = h_u * u
    F_v = h_v * v

    dhdt = -(ddx_u_to_c(F_u) + ddy_v_to_c(F_v)) + nu2_h * lap_c(h) + nu4_h * bih_c(h)

    return apply_bc_1l(du, dv, dhdt)

# -------------------------
# RK4
# -------------------------
def rk4_1l(u, v, h, dt):
    k1u, k1v, k1h = rhs_1l(u, v, h)

    ub, vb, hb = apply_bc_1l(u + 0.5 * dt * k1u, v + 0.5 * dt * k1v, h + 0.5 * dt * k1h)
    k2u, k2v, k2h = rhs_1l(ub, vb, hb)

    uc, vc, hc = apply_bc_1l(u + 0.5 * dt * k2u, v + 0.5 * dt * k2v, h + 0.5 * dt * k2h)
    k3u, k3v, k3h = rhs_1l(uc, vc, hc)

    ud, vd, hd = apply_bc_1l(u + dt * k3u, v + dt * k3v, h + dt * k3h)
    k4u, k4v, k4h = rhs_1l(ud, vd, hd)

    u_new = u + (dt / 6.0) * (k1u + 2 * k2u + 2 * k3u + k4u)
    v_new = v + (dt / 6.0) * (k1v + 2 * k2v + 2 * k3v + k4v)
    h_new = h + (dt / 6.0) * (k1h + 2 * k2h + 2 * k3h + k4h)

    return apply_bc_1l(u_new, v_new, h_new)

# -------------------------
# Collocation helpers
# -------------------------
def init_colloc_store():
    return {
        "t_sec": [],
        "fd_step": [],
        "macro_index": [],
        "ix": [],
        "iy": [],
        "x_m": [],
        "y_m": [],
        "eta": [],
        "uc": [],
        "vc": [],
        "f": [],
        "deta_dt_fd": [],
        "duc_dt_fd": [],
        "dvc_dt_fd": [],
        "etax_fd": [],
        "etay_fd": [],
        "ucx_fd": [],
        "ucy_fd": [],
        "vcx_fd": [],
        "vcy_fd": [],
        "div_fd": [],
        "zeta_fd": [],
    }

def finalize_colloc_store(store):
    out = {}
    for k, v in store.items():
        if len(v) == 0:
            if k in ("fd_step", "macro_index", "ix", "iy"):
                out[k] = np.array([], dtype=np.int32)
            else:
                out[k] = np.array([], dtype=np.float32)
        else:
            out[k] = np.concatenate(v, axis=0)
    return out

def save_colloc_npz(ic_key, store):
    os.makedirs(ROOT_COLLOC, exist_ok=True)
    out_path = os.path.join(ROOT_COLLOC, f"{ic_key}_colloc.npz")

    save_dict = finalize_colloc_store(store)
    save_dict["ic_key"] = np.array(ic_key)
    save_dict["nx"] = np.int32(nx)
    save_dict["ny"] = np.int32(ny)
    save_dict["dx"] = np.float32(dx)
    save_dict["dy"] = np.float32(dy)
    save_dict["dt"] = np.float32(dt)
    save_dict["H"] = np.float32(H)
    save_dict["fp"] = np.float32(fp)

    np.savez_compressed(out_path, **save_dict)
    return out_path

def sample_collocation_indices(weight, npts, rng):
    flat_n = ny * nx
    npts = min(npts, flat_n)

    if weight is None:
        idx = rng.choice(flat_n, size=npts, replace=False)
    else:
        w = np.asarray(weight, dtype=np.float64).reshape(-1)
        w = np.maximum(w, 0.0)
        s = w.sum()
        if s <= 0.0:
            idx = rng.choice(flat_n, size=npts, replace=False)
        else:
            w = w / s
            idx = rng.choice(flat_n, size=npts, replace=False, p=w)

    iy = idx // nx
    ix = idx % nx
    return iy.astype(np.int32), ix.astype(np.int32)

def colloc_weight_from_eta(eta, mode="grad_eta"):
    if mode == "uniform":
        return None
    if mode == "grad_eta":
        ex = ddx_center(eta)
        ey = ddy_center_edge(eta)
        w = np.sqrt(ex * ex + ey * ey)
        return w + 1.0e-12
    return None

def get_centered_state_and_fd_diagnostics(u, v, h):
    """
    Return centered state, centered FD time tendencies, and local centered
    FD spatial diagnostics, all on the (ny, nx) centered grid.

    State:
        eta, uc, vc

    Time tendencies:
        deta_dt_fd, duc_dt_fd, dvc_dt_fd

    Spatial diagnostics:
        etax_fd, etay_fd
        ucx_fd, ucy_fd
        vcx_fd, vcy_fd
        div_fd, zeta_fd
    """
    u, v, h = apply_bc_1l(u, v, h)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    du, dv, dhdt = rhs_1l(u, v, h)

    deta_dt_fd = dhdt
    duc_dt_fd = center_from_u(du)
    dvc_dt_fd = center_from_v(dv)

    etax_fd = ddx_center(eta)
    etay_fd = ddy_center_edge(eta)

    ucx_fd = ddx_center(uc)
    ucy_fd = ddy_center_edge(uc)

    vcx_fd = ddx_center(vc)
    vcy_fd = ddy_center_edge(vc)

    div_fd = ucx_fd + vcy_fd
    zeta_fd = vcx_fd - ucy_fd

    return (
        eta, uc, vc,
        deta_dt_fd, duc_dt_fd, dvc_dt_fd,
        etax_fd, etay_fd,
        ucx_fd, ucy_fd,
        vcx_fd, vcy_fd,
        div_fd, zeta_fd,
    )

def append_colloc_samples(store, step, macro_index, t, u, v, h, rng):
    (
        eta, uc, vc,
        deta_dt_fd, duc_dt_fd, dvc_dt_fd,
        etax_fd, etay_fd,
        ucx_fd, ucy_fd,
        vcx_fd, vcy_fd,
        div_fd, zeta_fd,
    ) = get_centered_state_and_fd_diagnostics(u, v, h)

    weight = colloc_weight_from_eta(eta, mode=COLLOC_WEIGHT_MODE)
    iy, ix = sample_collocation_indices(weight, N_COLLOC_PER_TIME, rng)

    store["t_sec"].append(np.full(len(iy), t, dtype=np.float32))
    store["fd_step"].append(np.full(len(iy), step, dtype=np.int32))
    store["macro_index"].append(np.full(len(iy), macro_index, dtype=np.int32))

    store["ix"].append(ix)
    store["iy"].append(iy)

    if SAVE_COLLOC_COORDS:
        store["x_m"].append(Xc[iy, ix].astype(np.float32))
        store["y_m"].append(Yc[iy, ix].astype(np.float32))
    else:
        store["x_m"].append(np.zeros(len(iy), dtype=np.float32))
        store["y_m"].append(np.zeros(len(iy), dtype=np.float32))

    store["eta"].append(eta[iy, ix].astype(np.float32))
    store["uc"].append(uc[iy, ix].astype(np.float32))
    store["vc"].append(vc[iy, ix].astype(np.float32))
    store["f"].append(f_c[iy, ix].astype(np.float32))

    store["deta_dt_fd"].append(deta_dt_fd[iy, ix].astype(np.float32))
    store["duc_dt_fd"].append(duc_dt_fd[iy, ix].astype(np.float32))
    store["dvc_dt_fd"].append(dvc_dt_fd[iy, ix].astype(np.float32))

    store["etax_fd"].append(etax_fd[iy, ix].astype(np.float32))
    store["etay_fd"].append(etay_fd[iy, ix].astype(np.float32))

    store["ucx_fd"].append(ucx_fd[iy, ix].astype(np.float32))
    store["ucy_fd"].append(ucy_fd[iy, ix].astype(np.float32))

    store["vcx_fd"].append(vcx_fd[iy, ix].astype(np.float32))
    store["vcy_fd"].append(vcy_fd[iy, ix].astype(np.float32))

    store["div_fd"].append(div_fd[iy, ix].astype(np.float32))
    store["zeta_fd"].append(zeta_fd[iy, ix].astype(np.float32))

# -------------------------
# Diagnostics / save
# -------------------------
def total_mass(h):
    return float(np.sum(h) * dx * dy)

def total_energy(u, v, h):
    uc = center_from_u(u)
    vc = center_from_v(v)
    eta = h - H
    ke = 0.5 * h * (uc**2 + vc**2)
    ape = 0.5* g/H *eta**2
    pe = 0.5 * g * (eta**2)
    return float(np.sum((ke + pe) * dx * dy))

def save_centered_1L(ic_key, step, u, v, h, t):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    os.makedirs(ic_dir, exist_ok=True)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    path = os.path.join(ic_dir, f"klein_step_{step:06d}.npz")
    np.savez_compressed(
        path,
        eta=eta.astype(np.float32),
        uc=uc.astype(np.float32),
        vc=vc.astype(np.float32),
        h=h.astype(np.float32),
        f=f_c.astype(np.float32),
        y_m=y_c.astype(np.float32),
        H=np.float32(H),
        dt=np.float32(dt),
        t=np.float32(t),
        nx=np.int32(nx), ny=np.int32(ny),
        dx=np.float32(dx), dy=np.float32(dy), fp=np.float32(fp),
    )
    return path

def quick_plot_1L(ic_key, step, u, v, h):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    pdir = os.path.join(ic_dir, "plots")
    os.makedirs(pdir, exist_ok=True)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    xkm = Xc / 1e3
    ykm = (Yc - 0.5 * Ly) / 1e3

    fig, axs = plt.subplots(1, 3, figsize=(14, 3.5))

    im = axs[0].pcolormesh(xkm, ykm, eta, shading="auto")
    axs[0].set_title("eta (m)")
    plt.colorbar(im, ax=axs[0])

    im = axs[1].pcolormesh(xkm, ykm, uc, shading="auto")
    axs[1].set_title("u_c (m/s)")
    plt.colorbar(im, ax=axs[1])

    im = axs[2].pcolormesh(xkm, ykm, vc, shading="auto")
    axs[2].set_title("v_c (m/s)")
    plt.colorbar(im, ax=axs[2])

    plt.tight_layout()
    plt.savefig(os.path.join(pdir, f"fields_step_{step:06d}.png"), dpi=120)
    plt.close()

def plot_mass_energy(ic_key, steps, m_series, e_series):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    pdir = os.path.join(ic_dir, "plots")
    os.makedirs(pdir, exist_ok=True)

    tdays = np.asarray(steps) * dt / 86400.0
    M0 = m_series[0]
    E0 = e_series[0]

    plt.figure(figsize=(8, 4))
    plt.plot(tdays, (np.asarray(m_series) - M0) / M0, label="ΔM/M0")
    plt.plot(tdays, (np.asarray(e_series) - E0) / E0, label="ΔE/E0")
    plt.xlabel("time (days)")
    plt.ylabel("relative change")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "mass_energy_timeseries.png"), dpi=120)
    plt.close()

# -------------------------
# Load bundle and IC keys
# -------------------------
def list_ic_keys(bundle_path):
    d = np.load(bundle_path)
    keys = sorted(set(k[:-2] for k in d.files if k.endswith("_h")))
    return keys

def load_ic(bundle_path, ic_key):
    d = np.load(bundle_path)
    h = d[f"{ic_key}_h"].astype(np.float32)
    u = d[f"{ic_key}_u"].astype(np.float32)
    v = d[f"{ic_key}_v"].astype(np.float32)
    return h, u, v

# -------------------------
# Run one IC
# -------------------------
def run_ic(ic_key):
    print(f"\n=== 1L IC: {ic_key} | nx={nx}, ny={ny}, dt={dt:.1f}s, nt={nt} ===")

    # Restart-safe skip: do not rerun completed ICs.
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    final_path = os.path.join(ic_dir, FINAL_SNAPSHOT)
    if SKIP_COMPLETED and os.path.exists(final_path):
        print(f"[skip] {ic_key} already completed: {final_path}")
        return

    h, u, v = load_ic(IC_BUNDLE, ic_key)
    u, v, h = apply_bc_1l(u, v, h)
    h[:] = np.maximum(h, 5.0)

    steps = []
    m_series = []
    e_series = []

    M0 = total_mass(h)
    E0 = total_energy(u, v, h)
    t = 0.0

    rng = np.random.default_rng(COLLOC_SEED)
    colloc_store = init_colloc_store()
    macro_index = 0

    steps.append(0)
    m_series.append(M0)
    e_series.append(E0)

    if 0 in SAVE_STEPS:
        p = save_centered_1L(ic_key, 0, u, v, h, t)
        quick_plot_1L(ic_key, 0, u, v, h)
        print(f"[save] {ic_key} step=0 -> {p}")
        macro_index += 1

    if SAVE_COLLOC and (0 % COLLOC_EVERY_N_STEPS == 0):
        append_colloc_samples(
            store=colloc_store,
            step=0,
            macro_index=macro_index,
            t=t,
            u=u, v=v, h=h,
            rng=rng,
        )

    for n in range(1, nt + 1):
        u, v, h = rk4_1l(u, v, h, dt)
        t += dt

        u, v, h, _ = enforce_floor_ke_preserving(u, v, h, HMIN)

        steps.append(n)
        m_series.append(total_mass(h))
        e_series.append(total_energy(u, v, h))

        if n in SAVE_STEPS:
            p = save_centered_1L(ic_key, n, u, v, h, t)
            quick_plot_1L(ic_key, n, u, v, h)
            print(f"[save] {ic_key} step={n} -> {p}")
            macro_index += 1

        if SAVE_COLLOC and (n % COLLOC_EVERY_N_STEPS == 0):
            append_colloc_samples(
                store=colloc_store,
                step=n,
                macro_index=macro_index,
                t=t,
                u=u, v=v, h=h,
                rng=rng,
            )

        if (n % 100) == 0 or n == 1:
            uc = center_from_u(u)
            vc = center_from_v(v)
            umax = float(np.max(np.sqrt(uc * uc + vc * vc)))
            dM = (m_series[-1] - M0) / M0
            dE = (e_series[-1] - E0) / E0
            print(f"[{n:5d}] dM/M0={dM:+.3e}  dE/E0={dE:+.3e}  umax={umax:6.2f}")

    plot_mass_energy(ic_key, steps, m_series, e_series)

    if SAVE_COLLOC:
        colloc_path = save_colloc_npz(ic_key, colloc_store)
        print(f"[colloc] {ic_key} -> {colloc_path}")

    print(f"Done (1L): {ic_key}")

# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    os.makedirs(ROOT_OUT, exist_ok=True)
    os.makedirs(ROOT_COLLOC, exist_ok=True)

    keys = list_ic_keys(IC_BUNDLE)
    print("ICs in bundle:", keys)

    # Colab-friendly incremental batch selection.
    end = BATCH_END if BATCH_END is not None else len(keys)
    run_keys = keys[BATCH_START:end]

    print(f"Running batch keys[{BATCH_START}:{end}] -> {len(run_keys)} ICs")
    print("Batch ICs:", run_keys)
    print("Output root:", ROOT_OUT)
    print("Collocation root:", ROOT_COLLOC)
    print("Save steps:", sorted(SAVE_STEPS))

    for k in run_keys:
        run_ic(k)

ICs in bundle: []
Running batch keys[20:40] -> 0 ICs
Batch ICs: []
Output root: /content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50plus
Collocation root: /content/drive/MyDrive/AI_EMUL3/klein_ml_1L_big50plus/colloc_raw
Save steps: [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000, 1050, 1100, 1150, 1200]


# Merge original and extended bundles in a unified folder

In [ ]:
import os
import glob
import shutil

# ============================================================
# INPUTS
# ============================================================

OLD_FD = "/content/drive/MyDrive/AI_EMUL/klein_ckpt_1L_big50"
NEW_FD = "/content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50plus"

OUT_FD = "/content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_combined"

OLD_COLLOC = "/content/drive/MyDrive/AI_EMUL/klein_ml_1L_big50/colloc_raw"

NEW_COLLOC = "/content/drive/MyDrive/AI_EMUL3/klein_ml_1L_big50plus/colloc_raw"

OUT_COLLOC = "/content/drive/MyDrive/AI_EMUL3/klein_ml_1L_big50_combined/colloc_raw"

# ============================================================
# Make output dirs
# ============================================================

os.makedirs(OUT_FD, exist_ok=True)
os.makedirs(OUT_COLLOC, exist_ok=True)

# ============================================================
# Merge FD trajectory folders
# ============================================================

def copy_ic_dirs(src_root, dst_root):
    ic_dirs = sorted([
        d for d in glob.glob(os.path.join(src_root, "*"))
        if os.path.isdir(d)
    ])

    for d in ic_dirs:
        name = os.path.basename(d)
        dst = os.path.join(dst_root, name)

        if os.path.exists(dst):
            print("Skipping existing:", name)
            continue

        print("Copying:", name)
        shutil.copytree(d, dst)

copy_ic_dirs(OLD_FD, OUT_FD)
copy_ic_dirs(NEW_FD, OUT_FD)

# ============================================================
# Merge collocation files
# ============================================================

def copy_colloc(src_root, dst_root):
    files = sorted(glob.glob(os.path.join(src_root, "*.npz")))

    for f in files:
        name = os.path.basename(f)
        dst = os.path.join(dst_root, name)

        if os.path.exists(dst):
            print("Skipping existing colloc:", name)
            continue

        print("Copying colloc:", name)
        shutil.copy2(f, dst)

copy_colloc(OLD_COLLOC, OUT_COLLOC)
copy_colloc(NEW_COLLOC, OUT_COLLOC)

print()
print("DONE")
print("Merged FD root:")
print(OUT_FD)
print()
print("Merged colloc root:")
print(OUT_COLLOC)

Copying: gauss_00
Copying: gauss_01
Copying: gauss_02
Copying: gauss_03
Copying: gauss_04
Copying: gauss_05
Copying: gauss_aug_000
Copying: gauss_aug_001
Copying: gauss_aug_002
Copying: gauss_aug_003
Copying: gauss_aug_004
Copying: gauss_aug_005
Copying: gauss_aug_006
Copying: gauss_aug_007
Copying: gauss_aug_008
Copying: gauss_aug_009
Copying: gauss_aug_010
Copying: gauss_aug_011
Copying: gauss_aug_012
Copying: gauss_aug_013
Copying: gauss_aug_014
Copying: gauss_aug_015
Copying: gauss_aug_016
Copying: gauss_aug_017
Copying: gauss_aug_018
Copying: gauss_aug_019
Copying: mix_00
Copying: mix_01
Copying: mix_02
Copying: mix_03
Copying: mix_04
Copying: mix_05
Copying: modon_aug_000
Copying: modon_aug_001
Copying: modon_aug_002
Copying: modon_aug_003
Copying: modon_aug_004
Copying: modon_aug_005
Copying: modon_aug_006
Copying: modon_aug_007
Copying: modon_aug_008
Copying: modon_aug_009
Copying: modon_aug_010
Copying: modon_aug_011
Copying: modon_aug_012
Copying: modon_aug_013
Copying: modon

# Emulator for extended data

In [ ]:
# ============================================================
# train_resunet_axial_1layer_big50_ai_emul3.py
# ------------------------------------------------------------
# ResUNet + axial-attention trainer for 1-layer SWE emulator
# on Klein beta-plane.
#
# New AI_EMUL3 branch:
#   - ResUNet multiscale backbone
#   - axial attention bottleneck for nonlocal wave transport
#   - Arakawa-inspired spectral UV / vorticity / KE regularization
#
# Outputs:
#   /content/drive/MyDrive/AI_EMUL3/training_runs/...
# ============================================================

import os, sys, glob, csv, time, random, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

sys.path.append("/content/drive/MyDrive/AI_4DVAR")
from colloc_bank import CollocBank

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

class CFG:
    ROOT_CODE = "/content/drive/MyDrive/AI_4DVAR"
    ROOT_OUT  = "/content/drive/MyDrive/AI_EMUL3"

    # ------------------------------------------------------------
    # Expanded BIG50PLUS dataset
    # ------------------------------------------------------------
    DATA_DIR_CANDIDATES = [
      "/content/AI_EMUL_LOCAL/klein_ckpt_1L_big50_combined",
      "/content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_combined",
    ]

    COLLOC_DIR_CANDIDATES = [
      "/content/AI_EMUL_LOCAL/klein_ml_1L_big50_combined/colloc_raw",
      "/content/drive/MyDrive/AI_EMUL3/klein_ml_1L_big50_combined/colloc_raw",
    ]

    # ------------------------------------------------------------
    # New expanded training experiment
    # ------------------------------------------------------------
    EXP_NAME = "resunet_axial_b96_big50plus_t8_arakawa_spec_lr5e5_e100"

    # ------------------------------------------------------------
    # Longer training
    # ------------------------------------------------------------
    EPOCHS = 100


    NX = 256
    NY = 128
    Lx = 2.0e7
    Ly = 8.0e6
    H = 1000.0
    G = 9.81
    DT_MACRO = 50.0 * 30.0
    FMIN = 2.0e-5

    U_SCALE = 50.0
    KE_SCALE = 0.5 * U_SCALE**2

    BASE_CH = 96
    FEAT_CH = 96
    HIDDEN = 192
    ENC_BLOCKS = 2
    MID_BLOCKS = 3
    DEC_BLOCKS = 2

    USE_AXIAL_ATTN = True
    AXIAL_HEADS = 4
    AXIAL_BLOCKS = 2

    BATCH_SIZE = 1
    LR = 5e-5
    WEIGHT_DECAY = 1e-6
    GRAD_CLIP = 1.0

    # Start conservatively for axial attention. Later try 6 or 8 if stable.
    ROLL_STEPS = 4
    ROLL_GAMMA = 0.95

    MAX_WINDOWS_PER_IC = 16
    NUM_WORKERS = 0
    PIN_MEMORY = torch.cuda.is_available()

    N_TIME_SLICES = 8
    PTS_PER_TIME = 4

    LAMBDA_DATA = 1.0
    LAMBDA_COLL_STATE = 0.1
    LAMBDA_COLL_TEND = 0.2
    LAMBDA_CONT = 0.5
    LAMBDA_MOM = 5.0
    LAMBDA_GEO = 0.01

    LAMBDA_UV_MAG = 1e-6
    LAMBDA_KE = 1e-4

    # Arakawa-inspired spectral controls
    LAMBDA_SPEC_UV = 2e-2
    LAMBDA_SPEC_ZETA = 2e-2
    LAMBDA_SPEC_KE = 1e-2
    LAMBDA_HIGHK = 1e-3
    HIGHK_FRAC = 0.55

    USE_STATE_CLAMP = True
    ETA_CLAMP = 2000.0
    UV_CLAMP = 150.0

    SAVE_EVERY_EPOCH = 1
    PRINT_BATCH_EVERY = 25
    AUTO_RESUME = True
    RESUME_FROM = None
    MAX_BATCHES_PER_EPOCH = None

cfg = CFG()
cfg.DX = cfg.Lx / cfg.NX
cfg.DY = cfg.Ly / cfg.NY
cfg.CKPT_DIR = os.path.join(cfg.ROOT_OUT, "training_runs", cfg.EXP_NAME)
os.makedirs(cfg.CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic, n_pairs = 0, 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += len(files) - 1
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir, best_pairs = None, -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_dir, best_pairs = d, n_pairs
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found. Check DATA_DIR_CANDIDATES.")
    print(f"[AutoDetect] using data: {best_dir}")
    return best_dir

def autodetect_existing_dir(candidates, label="directory"):
    print(f"[AutoDetect] checking candidate {label}s...")
    for d in candidates:
        print(f"  {d} -> exists={os.path.exists(d)}")
        if os.path.exists(d):
            print(f"[AutoDetect] using {label}: {d}")
            return d
    raise RuntimeError(f"No valid {label} found from candidates.")

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir, colloc_dir):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss_history": loss_history,
        "data_dir": data_dir,
        "colloc_dir": colloc_dir,
        "exp_name": cfg.EXP_NAME,
        "model_class": "MultiStepContinuousResUNetAxialModel",
        "config": {k: v for k, v in cfg.__dict__.items() if not k.startswith("__")},
    }, path)

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt.get("epoch", -1), ckpt.get("loss_history", []), ckpt

def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "epoch", "train_total", "train_data",
            "train_coll_state", "train_coll_tend",
            "train_cont", "train_mom", "train_geo",
            "train_uv_mag", "train_ke",
            "train_spec_uv", "train_spec_zeta", "train_spec_ke", "train_highk",
        ])
        for row in loss_history:
            w.writerow(row)

def clamp_state(x):
    if not cfg.USE_STATE_CLAMP:
        return x
    eta = torch.clamp(x[:, 0:1], -cfg.ETA_CLAMP, cfg.ETA_CLAMP)
    u = torch.clamp(x[:, 1:2], -cfg.UV_CLAMP, cfg.UV_CLAMP)
    v = torch.clamp(x[:, 2:3], -cfg.UV_CLAMP, cfg.UV_CLAMP)
    return torch.cat([eta, u, v], dim=1)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class SWWindowDataset(Dataset):
    def __init__(self, data_dir, roll_steps=4, max_windows_per_ic=None):
        self.samples = []
        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} IC directories")
        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")), key=extract_step_from_path)
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")
            if len(files) < roll_steps + 1:
                continue
            windows = []
            for i in range(len(files) - roll_steps):
                fseq = files[i:i + roll_steps + 1]
                macro_start_index = i + 1
                windows.append((fseq, ic_key, macro_start_index))
            if max_windows_per_ic is not None:
                windows = windows[:max_windows_per_ic]
            self.samples.extend(windows)
        if len(self.samples) == 0:
            raise RuntimeError("No usable training windows found.")
        print(f"[Dataset] total windows = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fseq, ic_key, macro_start_index = self.samples[idx]
        seq, times, steps = [], [], []
        for f in fseq:
            z = np.load(f)
            seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
            times.append(float(z["t"]))
            steps.append(int(extract_step_from_path(f)))
        return {
            "seq": torch.from_numpy(np.stack(seq, axis=0)),
            "times": np.array(times, dtype=np.float32),
            "steps": np.array(steps, dtype=np.int32),
            "ic_key": ic_key,
            "macro_start_index": np.int32(macro_start_index),
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, padding=dilation, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=dilation, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class ResStack(nn.Module):
    def __init__(self, ch, n_blocks=2, dilations=(1, 2)):
        super().__init__()
        self.net = nn.Sequential(*[ResBlock(ch, dilations[i % len(dilations)]) for i in range(n_blocks)])

    def forward(self, x):
        return self.net(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_blocks=2):
        super().__init__()
        self.down = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1), nn.GELU())
        self.res = ResStack(out_ch, n_blocks=n_blocks, dilations=(1, 2))

    def forward(self, x):
        return self.res(self.down(x))

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, n_blocks=2):
        super().__init__()
        self.up = nn.Sequential(nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2), nn.GELU())
        self.fuse = nn.Sequential(nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1), nn.GELU())
        self.res = ResStack(out_ch, n_blocks=n_blocks, dilations=(1, 2))

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.res(self.fuse(x))

class AxialAttentionBlock(nn.Module):
    def __init__(self, ch, heads=4):
        super().__init__()
        self.norm_x = nn.LayerNorm(ch)
        self.attn_x = nn.MultiheadAttention(ch, heads, batch_first=True)
        self.norm_y = nn.LayerNorm(ch)
        self.attn_y = nn.MultiheadAttention(ch, heads, batch_first=True)
        self.norm_ff = nn.LayerNorm(ch)
        self.ff = nn.Sequential(nn.Linear(ch, 4 * ch), nn.GELU(), nn.Linear(4 * ch, ch))

    def forward(self, x):
        B, C, H, W = x.shape
        xx = x.permute(0, 2, 3, 1).reshape(B * H, W, C)
        q = self.norm_x(xx)
        ax, _ = self.attn_x(q, q, q, need_weights=False)
        xx = xx + ax
        x = xx.reshape(B, H, W, C).permute(0, 3, 1, 2).contiguous()

        yy = x.permute(0, 3, 2, 1).reshape(B * W, H, C)
        q = self.norm_y(yy)
        ay, _ = self.attn_y(q, q, q, need_weights=False)
        yy = yy + ay
        q = self.norm_ff(yy)
        yy = yy + self.ff(q)
        return yy.reshape(B, W, H, C).permute(0, 3, 2, 1).contiguous()

class MultiStepContinuousResUNetAxialModel(nn.Module):
    def __init__(self, in_ch=3, base_ch=96, feat_ch=96, hidden=192,
                 enc_blocks=2, mid_blocks=3, dec_blocks=2,
                 axial_heads=4, axial_blocks=2, use_axial_attn=True):
        super().__init__()
        c0, c1, c2 = base_ch, base_ch * 2, base_ch * 4
        self.stem = nn.Sequential(nn.Conv2d(in_ch, c0, 3, padding=1), nn.GELU(), nn.Conv2d(c0, c0, 3, padding=1), nn.GELU())
        self.enc0 = ResStack(c0, n_blocks=enc_blocks, dilations=(1, 2))
        self.down1 = DownBlock(c0, c1, n_blocks=enc_blocks)
        self.down2 = DownBlock(c1, c2, n_blocks=enc_blocks)
        self.mid_pre = ResStack(c2, n_blocks=mid_blocks, dilations=(1, 2, 4))
        self.axial = nn.Sequential(*[AxialAttentionBlock(c2, heads=axial_heads) for _ in range(axial_blocks)]) if use_axial_attn else nn.Identity()
        self.mid_post = ResStack(c2, n_blocks=mid_blocks, dilations=(1, 2, 4))
        self.up1 = UpBlock(c2, c1, c1, n_blocks=dec_blocks)
        self.up0 = UpBlock(c1, c0, c0, n_blocks=dec_blocks)
        self.feat_proj = nn.Sequential(nn.Conv2d(c0, feat_ch, 3, padding=1), nn.GELU(), nn.Conv2d(feat_ch, feat_ch, 3, padding=1), nn.GELU())
        self.grid_head = nn.Sequential(nn.Conv2d(feat_ch, feat_ch, 3, padding=1), nn.GELU(), nn.Conv2d(feat_ch, 3, 3, padding=1))
        self.query_mlp = nn.Sequential(nn.Linear(feat_ch + 3 + 3, hidden), nn.GELU(), nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, 3))
        nn.init.zeros_(self.grid_head[-1].weight)
        nn.init.zeros_(self.grid_head[-1].bias)
        nn.init.zeros_(self.query_mlp[-1].weight)
        nn.init.zeros_(self.query_mlp[-1].bias)

    def encode(self, x):
        s0 = self.stem(x)
        e0 = self.enc0(s0)
        e1 = self.down1(e0)
        e2 = self.down2(e1)
        m = self.mid_pre(e2)
        m = self.axial(m)
        m = self.mid_post(m)
        d1 = self.up1(m, e1)
        d0 = self.up0(d1, e0)
        return self.feat_proj(d0)

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(field_1b, grid, mode="bilinear", padding_mode="border", align_corners=True)
        return vals.squeeze(0).squeeze(-1).transpose(0, 1)

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        feat_local = self._sample_local(feat_1b, x_norm.detach(), y_norm.detach())
        state0_local = self._sample_local(x_1b, x_norm.detach(), y_norm.detach())
        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        delta = self.query_mlp(torch.cat([feat_local, state0_local, q], dim=-1))
        return state0_local + tau.unsqueeze(-1) * delta

# ------------------------------------------------------------
# Collocation and losses
# ------------------------------------------------------------
def sample_multitime_colloc(colloc_bank, ic_key, macro_index, n_time_slices=4, pts_per_time=4):
    n_request = max(n_time_slices * pts_per_time * 8, 64)
    base = colloc_bank.sample_nearest_macro(ic_key=ic_key, macro_index=macro_index, npts=n_request, replace=True)
    t_sec = np.asarray(base["t_sec"]).reshape(-1)
    if t_sec.size == 0:
        return base
    order = np.argsort(t_sec)
    bins = np.array_split(order, n_time_slices)
    chosen = []
    for b in bins:
        if len(b) == 0:
            continue
        take = min(pts_per_time, len(b))
        chosen.append(np.random.choice(b, size=take, replace=False))
    if len(chosen) == 0:
        chosen = [order[:min(pts_per_time, len(order))]]
    chosen = np.concatenate(chosen, axis=0)
    out = {}
    for k, arr in base.items():
        arr = np.asarray(arr)
        out[k] = arr[chosen] if arr.ndim >= 1 and arr.shape[0] == len(t_sec) else arr
    return out

def safe_coriolis_torch(f, fmin):
    sign = torch.sign(f)
    sign = torch.where(sign == 0.0, torch.ones_like(sign), sign)
    return sign * torch.clamp(torch.abs(f), min=fmin)

def continuous_interval_losses_multitime(model, x_1b, feat_1b, colloc, t_start, dt_macro):
    x_m = torch.as_tensor(colloc["x_m"], dtype=torch.float32, device=device)
    y_m = torch.as_tensor(colloc["y_m"], dtype=torch.float32, device=device)
    t_sec = torch.as_tensor(colloc["t_sec"], dtype=torch.float32, device=device)
    tau = torch.clamp((t_sec - t_start) / dt_macro, 0.0, 1.0)
    x_norm = ((2.0 * x_m / cfg.Lx) - 1.0).clone().detach().requires_grad_(True)
    y_norm = ((2.0 * y_m / cfg.Ly) - 1.0).clone().detach().requires_grad_(True)
    tau = tau.clone().detach().requires_grad_(True)

    state_hat = model.query_state(x_1b, feat_1b, x_norm, y_norm, tau)
    eta_hat, u_hat, v_hat = state_hat[:, 0], state_hat[:, 1], state_hat[:, 2]
    h_hat = cfg.H + eta_hat

    def grads_of(field):
        return torch.autograd.grad(field.sum(), [x_norm, y_norm, tau], create_graph=True, retain_graph=True)

    eta_xn, eta_yn, eta_tau = grads_of(eta_hat)
    u_xn, u_yn, u_tau = grads_of(u_hat)
    v_xn, v_yn, v_tau = grads_of(v_hat)
    eta_x, eta_y = eta_xn * (2.0 / cfg.Lx), eta_yn * (2.0 / cfg.Ly)
    u_x, u_y = u_xn * (2.0 / cfg.Lx), u_yn * (2.0 / cfg.Ly)
    v_x, v_y = v_xn * (2.0 / cfg.Lx), v_yn * (2.0 / cfg.Ly)
    eta_t, u_t, v_t = eta_tau / dt_macro, u_tau / dt_macro, v_tau / dt_macro

    hu = h_hat * u_hat
    hv = h_hat * v_hat
    hu_xn, _, _ = grads_of(hu)
    _, hv_yn, _ = grads_of(hv)
    hu_x, hv_y = hu_xn * (2.0 / cfg.Lx), hv_yn * (2.0 / cfg.Ly)

    eta_true = torch.as_tensor(colloc["eta"], dtype=torch.float32, device=device)
    u_true = torch.as_tensor(colloc["uc"], dtype=torch.float32, device=device)
    v_true = torch.as_tensor(colloc["vc"], dtype=torch.float32, device=device)
    deta_dt_fd = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=device)
    duc_dt_fd = torch.as_tensor(colloc["duc_dt_fd"], dtype=torch.float32, device=device)
    dvc_dt_fd = torch.as_tensor(colloc["dvc_dt_fd"], dtype=torch.float32, device=device)
    f_fd = torch.as_tensor(colloc["f"], dtype=torch.float32, device=device)

    loss_coll_state = ((eta_hat - eta_true) ** 2).mean() + ((u_hat - u_true) ** 2).mean() + ((v_hat - v_true) ** 2).mean()
    loss_coll_tend = ((eta_t - deta_dt_fd) ** 2).mean() + ((u_t - duc_dt_fd) ** 2).mean() + ((v_t - dvc_dt_fd) ** 2).mean()
    loss_cont = (eta_t + hu_x + hv_y).pow(2).mean()

    adv_u = u_hat * u_x + v_hat * u_y
    adv_v = u_hat * v_x + v_hat * v_y
    resid_u = u_t + adv_u - f_fd * v_hat + cfg.G * eta_x
    resid_v = v_t + adv_v + f_fd * u_hat + cfg.G * eta_y
    loss_mom = ((resid_u / (1.0 + torch.abs(u_hat))) ** 2).mean() + ((resid_v / (1.0 + torch.abs(v_hat))) ** 2).mean()

    f_safe = safe_coriolis_torch(f_fd, cfg.FMIN)
    u_g = -(cfg.G / f_safe) * eta_y
    v_g = (cfg.G / f_safe) * eta_x
    loss_geo = ((u_hat - u_g) ** 2).mean() + ((v_hat - v_g) ** 2).mean()

    u_nd, v_nd = u_hat / cfg.U_SCALE, v_hat / cfg.U_SCALE
    loss_uv_mag = (u_nd ** 2 + v_nd ** 2).mean()

    ke_hat = 0.5 * (u_hat ** 2 + v_hat ** 2)
    ke_true = 0.5 * (u_true ** 2 + v_true ** 2)
    loss_ke = (((ke_hat - ke_true) / cfg.KE_SCALE) ** 2).mean()
    return loss_coll_state, loss_coll_tend, loss_cont, loss_mom, loss_geo, loss_uv_mag, loss_ke

# ------------------------------------------------------------
# Spectral / Arakawa-inspired full-grid losses
# ------------------------------------------------------------
def fd_grad_x(q):
    return (torch.roll(q, shifts=-1, dims=-1) - torch.roll(q, shifts=1, dims=-1)) / (2.0 * cfg.DX)

def fd_grad_y(q):
    qp = torch.empty_like(q)
    qm = torch.empty_like(q)
    qp[:, :, :-1, :] = q[:, :, 1:, :]
    qp[:, :, -1, :] = q[:, :, -1, :]
    qm[:, :, 1:, :] = q[:, :, :-1, :]
    qm[:, :, 0, :] = q[:, :, 0, :]
    return (qp - qm) / (2.0 * cfg.DY)

def vorticity_grid(x):
    return fd_grad_x(x[:, 2:3]) - fd_grad_y(x[:, 1:2])

def spectral_masks(ny, nx, device):
    ky = torch.fft.fftfreq(ny, d=cfg.DY).to(device)[:, None]
    kx = torch.fft.rfftfreq(nx, d=cfg.DX).to(device)[None, :]
    kr = torch.sqrt(kx ** 2 + ky ** 2)
    kr_norm = kr / (kr.max() + 1e-12)
    high = (kr_norm >= cfg.HIGHK_FRAC).float()
    weight_k2 = kr_norm ** 2
    return kr_norm[None, None], high[None, None], weight_k2[None, None]

def spectral_arakawa_losses(pred, truth):
    B, C, NY, NX = pred.shape
    u_p, v_p = pred[:, 1:2], pred[:, 2:3]
    u_t, v_t = truth[:, 1:2], truth[:, 2:3]
    kr_norm, high, weight_k2 = spectral_masks(NY, NX, pred.device)

    Uh_p, Vh_p = torch.fft.rfft2(u_p, norm="ortho"), torch.fft.rfft2(v_p, norm="ortho")
    Uh_t, Vh_t = torch.fft.rfft2(u_t, norm="ortho"), torch.fft.rfft2(v_t, norm="ortho")
    KEp = Uh_p.abs() ** 2 + Vh_p.abs() ** 2
    KEt = Uh_t.abs() ** 2 + Vh_t.abs() ** 2
    denom_ke = KEt.mean().detach() + 1e-8

    loss_spec_uv = (((KEp - KEt) / denom_ke) ** 2).mean()
    loss_spec_ke = (weight_k2 * ((KEp - KEt) / denom_ke) ** 2).mean()

    Zh_p = torch.fft.rfft2(vorticity_grid(pred), norm="ortho")
    Zh_t = torch.fft.rfft2(vorticity_grid(truth), norm="ortho")
    ZEp, ZEt = Zh_p.abs() ** 2, Zh_t.abs() ** 2
    denom_z = ZEt.mean().detach() + 1e-12
    loss_spec_zeta = (((ZEp - ZEt) / denom_z) ** 2).mean()
    loss_highk = (high * KEp / denom_ke).mean() + (high * ZEp / denom_z).mean()
    return loss_spec_uv, loss_spec_zeta, loss_spec_ke, loss_highk

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(cfg.DATA_DIR_CANDIDATES)
    colloc_dir = autodetect_existing_dir(cfg.COLLOC_DIR_CANDIDATES, label="collocation directory")
    colloc_bank = CollocBank(colloc_dir, verbose=True)
    dataset = SWWindowDataset(data_dir, roll_steps=cfg.ROLL_STEPS, max_windows_per_ic=cfg.MAX_WINDOWS_PER_IC)
    loader = DataLoader(dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY, drop_last=False)

    model = MultiStepContinuousResUNetAxialModel(
        base_ch=cfg.BASE_CH, feat_ch=cfg.FEAT_CH, hidden=cfg.HIDDEN,
        enc_blocks=cfg.ENC_BLOCKS, mid_blocks=cfg.MID_BLOCKS, dec_blocks=cfg.DEC_BLOCKS,
        axial_heads=cfg.AXIAL_HEADS, axial_blocks=cfg.AXIAL_BLOCKS, use_axial_attn=cfg.USE_AXIAL_ATTN
    ).to(device)
    print(f"[Model] trainable parameters = {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

    start_epoch, loss_history = 0, []
    resume_path = cfg.RESUME_FROM
    auto_last = os.path.join(cfg.CKPT_DIR, "last_model.pt")
    if resume_path is None and cfg.AUTO_RESUME and os.path.exists(auto_last):
        resume_path = auto_last
    if resume_path is not None and os.path.exists(resume_path):
        print(f"[Resume] loading checkpoint: {resume_path}")
        loaded_epoch, loaded_history, _ = load_checkpoint(resume_path, model, optimizer)
        start_epoch, loss_history = loaded_epoch + 1, loaded_history
        print(f"[Resume] continuing from epoch {start_epoch}")
    else:
        print("[Resume] no checkpoint found; starting from scratch")

    best_total = min([row[1] for row in loss_history], default=float("inf"))
    if start_epoch >= cfg.EPOCHS:
        print(f"[Train] checkpoint already reached epoch {start_epoch-1}; EPOCHS={cfg.EPOCHS}. Nothing to do.")
        return

    print(f"[Train] output dir            = {cfg.CKPT_DIR}")
    print(f"[Train] data dir              = {data_dir}")
    print(f"[Train] colloc dir            = {colloc_dir}")
    print(f"[Train] windows               = {len(dataset)}")
    print(f"[Train] batches/epoch         = {len(loader)}")
    print(f"[Train] ROLL_STEPS            = {cfg.ROLL_STEPS}")
    print(f"[Train] EPOCHS                = {cfg.EPOCHS}")
    print(f"[Train] AXIAL_HEADS/BLOCKS    = {cfg.AXIAL_HEADS}/{cfg.AXIAL_BLOCKS}")

    for epoch in range(start_epoch, cfg.EPOCHS):
        t0 = time.time()
        model.train()
        run = {k: 0.0 for k in ["total", "data", "cstate", "ctend", "cont", "mom", "geo", "uv_mag", "ke", "spec_uv", "spec_zeta", "spec_ke", "highk"]}

        for ib, batch in enumerate(loader):
            if cfg.MAX_BATCHES_PER_EPOCH is not None and ib >= cfg.MAX_BATCHES_PER_EPOCH:
                print(f"[debug] stopping epoch early at batch {ib}")
                break
            seq = batch["seq"].to(device, non_blocking=True)
            times = batch["times"].numpy()
            ic_keys = batch["ic_key"]
            macro_start = batch["macro_start_index"].numpy()
            B = seq.shape[0]
            optimizer.zero_grad(set_to_none=True)

            loss_data = 0.0
            loss_cstate = loss_ctend = loss_cont = loss_mom = loss_geo = loss_uv_mag = loss_ke = 0.0
            loss_spec_uv = loss_spec_zeta = loss_spec_ke = loss_highk = 0.0
            n_phys = 0
            x = seq[:, 0]

            for k in range(cfg.ROLL_STEPS):
                feat = model.encode(x)
                xdot_grid = model.grid_tendency(feat)
                x_next = clamp_state(x + cfg.DT_MACRO * xdot_grid)
                truth_next = seq[:, k + 1]

                loss_data = loss_data + (cfg.ROLL_GAMMA ** k) * ((x_next - truth_next) ** 2).mean()
                l_suv, l_szeta, l_ske, l_highk = spectral_arakawa_losses(x_next, truth_next)
                loss_spec_uv += l_suv
                loss_spec_zeta += l_szeta
                loss_spec_ke += l_ske
                loss_highk += l_highk

                for b in range(B):
                    ic_key = ic_keys[b]
                    macro_idx = int(macro_start[b] + k)
                    t_start = float(times[b, k])
                    if not colloc_bank.has_ic(ic_key):
                        continue
                    colloc = sample_multitime_colloc(colloc_bank, ic_key, macro_idx, cfg.N_TIME_SLICES, cfg.PTS_PER_TIME)
                    l_cstate, l_ctend, l_cont, l_mom, l_geo, l_uv_mag, l_ke = continuous_interval_losses_multitime(
                        model=model, x_1b=x[b:b+1], feat_1b=feat[b:b+1], colloc=colloc, t_start=t_start, dt_macro=cfg.DT_MACRO
                    )
                    loss_cstate += l_cstate
                    loss_ctend += l_ctend
                    loss_cont += l_cont
                    loss_mom += l_mom
                    loss_geo += l_geo
                    loss_uv_mag += l_uv_mag
                    loss_ke += l_ke
                    n_phys += 1
                x = x_next

            loss_spec_uv /= cfg.ROLL_STEPS
            loss_spec_zeta /= cfg.ROLL_STEPS
            loss_spec_ke /= cfg.ROLL_STEPS
            loss_highk /= cfg.ROLL_STEPS

            if n_phys > 0:
                loss_cstate /= n_phys
                loss_ctend /= n_phys
                loss_cont /= n_phys
                loss_mom /= n_phys
                loss_geo /= n_phys
                loss_uv_mag /= n_phys
                loss_ke /= n_phys
            else:
                zero = torch.tensor(0.0, device=device)
                loss_cstate = loss_ctend = loss_cont = loss_mom = loss_geo = loss_uv_mag = loss_ke = zero

            loss = (
                cfg.LAMBDA_DATA * loss_data
                + cfg.LAMBDA_COLL_STATE * loss_cstate
                + cfg.LAMBDA_COLL_TEND * loss_ctend
                + cfg.LAMBDA_CONT * loss_cont
                + cfg.LAMBDA_MOM * loss_mom
                + cfg.LAMBDA_GEO * loss_geo
                + cfg.LAMBDA_UV_MAG * loss_uv_mag
                + cfg.LAMBDA_KE * loss_ke
                + cfg.LAMBDA_SPEC_UV * loss_spec_uv
                + cfg.LAMBDA_SPEC_ZETA * loss_spec_zeta
                + cfg.LAMBDA_SPEC_KE * loss_spec_ke
                + cfg.LAMBDA_HIGHK * loss_highk
            )

            if not torch.isfinite(loss):
                print("Non-finite diagnostics:")
                for name, val in [
                    ("data", loss_data), ("cstate", loss_cstate), ("ctend", loss_ctend), ("cont", loss_cont),
                    ("mom", loss_mom), ("geo", loss_geo), ("uv_mag", loss_uv_mag), ("ke", loss_ke),
                    ("spec_uv", loss_spec_uv), ("spec_zeta", loss_spec_zeta), ("spec_ke", loss_spec_ke), ("highk", loss_highk)
                ]:
                    print(f"  {name:10s}=", float(val.detach().cpu()))
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

            vals = {
                "total": loss, "data": loss_data, "cstate": loss_cstate, "ctend": loss_ctend, "cont": loss_cont,
                "mom": loss_mom, "geo": loss_geo, "uv_mag": loss_uv_mag, "ke": loss_ke,
                "spec_uv": loss_spec_uv, "spec_zeta": loss_spec_zeta, "spec_ke": loss_spec_ke, "highk": loss_highk,
            }
            for kk, vv in vals.items():
                run[kk] += float(vv.detach().cpu())

            if (ib % cfg.PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={float(loss.detach().cpu()):.6e} data={float(loss_data.detach().cpu()):.6e} "
                    f"cstate={float(loss_cstate.detach().cpu()):.6e} ctend={float(loss_ctend.detach().cpu()):.6e} "
                    f"cont={float(loss_cont.detach().cpu()):.6e} mom={float(loss_mom.detach().cpu()):.6e} "
                    f"specuv={float(loss_spec_uv.detach().cpu()):.6e} specz={float(loss_spec_zeta.detach().cpu()):.6e} "
                    f"specke={float(loss_spec_ke.detach().cpu()):.6e} highk={float(loss_highk.detach().cpu()):.6e}"
                )

        n_batches = max(min(len(loader), cfg.MAX_BATCHES_PER_EPOCH) if cfg.MAX_BATCHES_PER_EPOCH is not None else len(loader), 1)
        ep = {k: v / n_batches for k, v in run.items()}
        loss_history.append([
            epoch, ep["total"], ep["data"], ep["cstate"], ep["ctend"], ep["cont"], ep["mom"], ep["geo"],
            ep["uv_mag"], ep["ke"], ep["spec_uv"], ep["spec_zeta"], ep["spec_ke"], ep["highk"],
        ])

        print(
            f"Epoch {epoch:03d} done | total={ep['total']:.6e} | data={ep['data']:.6e} | "
            f"cstate={ep['cstate']:.6e} | ctend={ep['ctend']:.6e} | cont={ep['cont']:.6e} | mom={ep['mom']:.6e} | "
            f"geo={ep['geo']:.6e} | uv_mag={ep['uv_mag']:.6e} | ke={ep['ke']:.6e} | "
            f"specuv={ep['spec_uv']:.6e} | specz={ep['spec_zeta']:.6e} | specke={ep['spec_ke']:.6e} | highk={ep['highk']:.6e} | "
            f"time={time.time() - t0:.1f}s"
        )

        last_ckpt = os.path.join(cfg.CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir, colloc_dir)
        if ((epoch + 1) % cfg.SAVE_EVERY_EPOCH) == 0:
            save_checkpoint(os.path.join(cfg.CKPT_DIR, f"model_epoch_{epoch:03d}.pt"), model, optimizer, epoch, loss_history, data_dir, colloc_dir)
        save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)

        if ep["total"] < best_total:
            best_total = ep["total"]
            best_ckpt = os.path.join(cfg.CKPT_DIR, "best_model.pt")
            save_checkpoint(best_ckpt, model, optimizer, epoch, loss_history, data_dir, colloc_dir)
            print(f"[Best] epoch={epoch:03d} train_total={ep['total']:.6e} -> {best_ckpt}")

    final_ckpt = os.path.join(cfg.CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, cfg.EPOCHS - 1, loss_history, data_dir, colloc_dir)
    save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)
    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")

if __name__ == "__main__":
    train()


Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[AutoDetect] checking candidate snapshot roots...
  /content/AI_EMUL_LOCAL/klein_ckpt_1L_big50_combined
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_combined
     valid IC dirs = 68, total pairs = 1632
[AutoDetect] using data: /content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_combined
[AutoDetect] checking candidate collocation directorys...
  /content/AI_EMUL_LOCAL/klein_ml_1L_big50_combined/colloc_raw -> exists=False
  /content/drive/MyDrive/AI_EMUL3/klein_ml_1L_big50_combined/colloc_raw -> exists=True
[AutoDetect] using collocation directory: /content/drive/MyDrive/AI_EMUL3/klein_ml_1L_big50_combined/colloc_raw
[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=25
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=25
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=25
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=25
[Collo

# New Section

In [ ]:
!find /content/drive/MyDrive -name "colloc_bank.py"

/content/drive/MyDrive/AI_4DVAR/colloc_bank.py


# Evaluation of Emulator


In [2]:
# ============================================================
# eval_resunet_axial_1layer_big50_ai_emul3.py
# ------------------------------------------------------------
# Evaluate trained ResUNet + axial-attention emulator
# on Gauss / Wave / RH cases.
#
# Matches:
#   train_resunet_axial_1layer_big50_ai_emul3.py
#
# Saves:
#   - summary_eval.csv
#   - all_rollout_metrics.csv
#   - per-IC rollout_metrics.csv
#   - RMSE plots
#   - KE plots
#   - final eta/u/v comparison plots
# ============================================================

import os, glob, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
class CFG:

    ROOT_DATA_CANDIDATES = [
        "/content/AI_EMUL_LOCAL",
        "/content/drive/MyDrive/AI_EMUL3",
    ]

    ROOT_RUN = "/content/drive/MyDrive/AI_EMUL3"

    RUN_NAME = "resunet_axial_b96_big50plus_t8_arakawa_spec_lr5e5_e100"

    RUN_DIR   = f"{ROOT_RUN}/training_runs/{RUN_NAME}"

    # Usually best_model.pt is preferred.
    # If training is interrupted and best_model.pt does not exist, use last_model.pt.
    CKPT_NAME = "best_model.pt"

    CKPT_PATH = f"{RUN_DIR}/{CKPT_NAME}"

    OUT_DIR = f"{RUN_DIR}/evaluation_gauss_wave_rh_raw_diff"


    NX = 256
    NY = 128
    Lx = 2.0e7
    Ly = 8.0e6
    H  = 1000.0
    DT_MACRO = 50.0 * 30.0

    # -------- optional inference diffusion --------
    # First evaluate without inference diffusion.
    #USE_INFERENCE_DIFFUSION = False
    #NU_INFER = 0.0

    # If later needed:
    USE_INFERENCE_DIFFUSION = True
    NU_INFER = 5.0e4

    DIFFUSE_ETA = False
    DIFFUSE_UV  = True

    # -------- architecture; must match training script --------
    BASE_CH = 96
    FEAT_CH = 96
    HIDDEN  = 192

    ENC_BLOCKS = 2
    MID_BLOCKS = 3
    DEC_BLOCKS = 2

    USE_AXIAL_ATTN = True
    AXIAL_HEADS = 4
    AXIAL_BLOCKS = 2

    # Evaluate these IC-name patterns
    IC_PATTERNS = ["gauss", "wave", "rh"]

    # None = use full available trajectory
    MAX_ROLLOUT_STEPS = None


cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# DATA AUTODETECT
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += len(files) - 1
    return n_ic, n_pairs


def autodetect_data_dir():
    candidates = [f"{root}/klein_ckpt_1L_big50_combined" for root in cfg.ROOT_DATA_CANDIDATES]

    print("[AutoDetect] checking data roots...")
    best_dir = None
    best_pairs = -1

    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d

    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid DATA_DIR found.")

    print("[AutoDetect] using DATA_DIR:", best_dir)
    return best_dir


DATA_DIR = autodetect_data_dir()

# ------------------------------------------------------------
# MODEL DEFINITION — must match AI_EMUL3 training script
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)


class ResStack(nn.Module):
    def __init__(self, ch, n_blocks=2, dilations=(1, 2)):
        super().__init__()
        self.net = nn.Sequential(*[
            ResBlock(ch, dilation=dilations[i % len(dilations)])
            for i in range(n_blocks)
        ])

    def forward(self, x):
        return self.net(x)


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_blocks=2):
        super().__init__()
        self.down = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1),
            nn.GELU(),
        )
        self.res = ResStack(out_ch, n_blocks=n_blocks, dilations=(1, 2))

    def forward(self, x):
        return self.res(self.down(x))


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, n_blocks=2):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2),
            nn.GELU(),
        )
        self.fuse = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=3, padding=1),
            nn.GELU(),
        )
        self.res = ResStack(out_ch, n_blocks=n_blocks, dilations=(1, 2))

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.res(self.fuse(x))


class AxialAttentionBlock(nn.Module):
    """
    Axial attention over x and y separately.
    Input/output: [B, C, H, W]
    """
    def __init__(self, ch, heads=4):
        super().__init__()
        self.ch = ch
        self.heads = heads

        self.norm_x = nn.LayerNorm(ch)
        self.attn_x = nn.MultiheadAttention(ch, heads, batch_first=True)

        self.norm_y = nn.LayerNorm(ch)
        self.attn_y = nn.MultiheadAttention(ch, heads, batch_first=True)

        self.norm_ff = nn.LayerNorm(ch)
        self.ff = nn.Sequential(
            nn.Linear(ch, 4 * ch),
            nn.GELU(),
            nn.Linear(4 * ch, ch),
        )

    def forward(self, x):
        B, C, H, W = x.shape

        # Attention along x for each y row.
        xx = x.permute(0, 2, 3, 1).reshape(B * H, W, C)
        q = self.norm_x(xx)
        ax, _ = self.attn_x(q, q, q, need_weights=False)
        xx = xx + ax
        x = xx.reshape(B, H, W, C).permute(0, 3, 1, 2).contiguous()

        # Attention along y for each x column.
        yy = x.permute(0, 3, 2, 1).reshape(B * W, H, C)
        q = self.norm_y(yy)
        ay, _ = self.attn_y(q, q, q, need_weights=False)
        yy = yy + ay

        # Feed-forward on column tokens.
        q = self.norm_ff(yy)
        yy = yy + self.ff(q)

        x = yy.reshape(B, W, H, C).permute(0, 3, 2, 1).contiguous()
        return x


class MultiStepContinuousResUNetAxialModel(nn.Module):
    def __init__(
        self,
        in_ch=3,
        base_ch=96,
        feat_ch=96,
        hidden=192,
        enc_blocks=2,
        mid_blocks=3,
        dec_blocks=2,
        axial_heads=4,
        axial_blocks=2,
        use_axial_attn=True,
    ):
        super().__init__()

        c0 = base_ch
        c1 = base_ch * 2
        c2 = base_ch * 4

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, c0, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(c0, c0, 3, padding=1),
            nn.GELU(),
        )

        self.enc0 = ResStack(c0, n_blocks=enc_blocks, dilations=(1, 2))
        self.down1 = DownBlock(c0, c1, n_blocks=enc_blocks)
        self.down2 = DownBlock(c1, c2, n_blocks=enc_blocks)

        self.mid_pre = ResStack(c2, n_blocks=mid_blocks, dilations=(1, 2, 4))

        if use_axial_attn:
            self.axial = nn.Sequential(*[
                AxialAttentionBlock(c2, heads=axial_heads)
                for _ in range(axial_blocks)
            ])
        else:
            self.axial = nn.Identity()

        self.mid_post = ResStack(c2, n_blocks=mid_blocks, dilations=(1, 2, 4))

        self.up1 = UpBlock(c2, c1, c1, n_blocks=dec_blocks)
        self.up0 = UpBlock(c1, c0, c0, n_blocks=dec_blocks)

        self.feat_proj = nn.Sequential(
            nn.Conv2d(c0, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        # Included for checkpoint compatibility with training.
        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        s0 = self.stem(x)
        e0 = self.enc0(s0)
        e1 = self.down1(e0)
        e2 = self.down2(e1)

        m = self.mid_pre(e2)
        m = self.axial(m)
        m = self.mid_post(m)

        d1 = self.up1(m, e1)
        d0 = self.up0(d1, e0)

        return self.feat_proj(d0)

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def forward_one_step(self, x, dt_macro):
        feat = self.encode(x)
        xdot = self.grid_tendency(feat)
        return x + dt_macro * xdot


# ------------------------------------------------------------
# LOAD MODEL
# ------------------------------------------------------------
model = MultiStepContinuousResUNetAxialModel(
    base_ch=cfg.BASE_CH,
    feat_ch=cfg.FEAT_CH,
    hidden=cfg.HIDDEN,
    enc_blocks=cfg.ENC_BLOCKS,
    mid_blocks=cfg.MID_BLOCKS,
    dec_blocks=cfg.DEC_BLOCKS,
    axial_heads=cfg.AXIAL_HEADS,
    axial_blocks=cfg.AXIAL_BLOCKS,
    use_axial_attn=cfg.USE_AXIAL_ATTN,
).to(device)

print("[Load] checkpoint:", cfg.CKPT_PATH)
ckpt = torch.load(cfg.CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("[Loaded checkpoint]")
print("checkpoint epoch:", ckpt.get("epoch", "unknown"))
print("model_class:", ckpt.get("model_class", "unknown"))

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
_step_re = re.compile(r"klein_step_(\d+)\.npz")


def step_from_file(path):
    m = _step_re.search(os.path.basename(path))
    return int(m.group(1)) if m is not None else -1


def load_state(path):
    z = np.load(path)
    eta = z["eta"].astype(np.float32)
    u   = z["uc"].astype(np.float32)
    v   = z["vc"].astype(np.float32)
    t   = float(z["t"]) if "t" in z else float(step_from_file(path) * 30.0)
    return np.stack([eta, u, v], axis=0), t


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def kinetic_energy(state):
    eta, u, v = state[0], state[1], state[2]
    h = cfg.H + eta
    return float(np.mean(0.5 * h * (u*u + v*v)))


def laplacian_periodic_x_reflect_y(q):
    q_xp = torch.roll(q, shifts=-1, dims=-1)
    q_xm = torch.roll(q, shifts= 1, dims=-1)

    q_yp = torch.empty_like(q)
    q_ym = torch.empty_like(q)

    q_yp[:, :, :-1, :] = q[:, :, 1:, :]
    q_yp[:, :, -1,  :] = q[:, :, -1, :]

    q_ym[:, :, 1:, :] = q[:, :, :-1, :]
    q_ym[:, :, 0,  :] = q[:, :, 0, :]

    return (
        (q_xp - 2.0*q + q_xm) / (cfg.Lx / cfg.NX)**2
        +
        (q_yp - 2.0*q + q_ym) / (cfg.Ly / cfg.NY)**2
    )


def apply_inference_diffusion(x):
    if not cfg.USE_INFERENCE_DIFFUSION:
        return x

    x_new = x.clone()

    if cfg.DIFFUSE_ETA:
        eta = x[:, 0:1]
        x_new[:, 0:1] = eta + cfg.NU_INFER * cfg.DT_MACRO * laplacian_periodic_x_reflect_y(eta)

    if cfg.DIFFUSE_UV:
        uv = x[:, 1:3]
        x_new[:, 1:3] = uv + cfg.NU_INFER * cfg.DT_MACRO * laplacian_periodic_x_reflect_y(uv)

    return x_new


def find_ic_dirs():
    all_dirs = sorted([d for d in glob.glob(os.path.join(DATA_DIR, "*")) if os.path.isdir(d)])
    selected = []

    for pat in cfg.IC_PATTERNS:
        matches = [d for d in all_dirs if pat.lower() in os.path.basename(d).lower()]
        if len(matches) == 0:
            print(f"[Warning] no IC matched pattern: {pat}")
        else:
            selected.extend(matches)

    seen, unique = set(), []
    for d in selected:
        if d not in seen:
            unique.append(d)
            seen.add(d)
    return unique


def plot_rmse(df_ic, ic_key, outdir):
    plt.figure(figsize=(8, 5))
    plt.plot(df_ic["lead_index"], df_ic["eta_rmse"], label="eta")
    plt.plot(df_ic["lead_index"], df_ic["u_rmse"], label="u")
    plt.plot(df_ic["lead_index"], df_ic["v_rmse"], label="v")
    plt.xlabel("Lead index, 50-step macro intervals")
    plt.ylabel("RMSE")
    plt.title(f"RMSE rollout: {ic_key}")
    plt.grid(True)
    plt.legend()
    path = os.path.join(outdir, f"{ic_key}_rmse.png")
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()
    return path


def plot_ke(df_ic, ic_key, outdir):
    plt.figure(figsize=(8, 5))
    plt.plot(df_ic["lead_index"], df_ic["fd_ke"], label="FD KE")
    plt.plot(df_ic["lead_index"], df_ic["ml_ke"], label="ML KE")
    plt.xlabel("Lead index, 50-step macro intervals")
    plt.ylabel("domain-mean KE")
    plt.title(f"KE rollout: {ic_key}")
    plt.grid(True)
    plt.legend()
    path1 = os.path.join(outdir, f"{ic_key}_ke.png")
    plt.savefig(path1, dpi=160, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(df_ic["lead_index"], df_ic["ke_ratio"])
    plt.axhline(1.0, linestyle="--")
    plt.xlabel("Lead index, 50-step macro intervals")
    plt.ylabel("ML KE / FD KE")
    plt.title(f"KE ratio: {ic_key}")
    plt.grid(True)
    path2 = os.path.join(outdir, f"{ic_key}_ke_ratio.png")
    plt.savefig(path2, dpi=160, bbox_inches="tight")
    plt.close()
    return path1, path2


def plot_final_fields(truth, pred, ic_key, lead_index, outdir):
    names = ["eta", "u", "v"]

    for c, name in enumerate(names):
        err = pred[c] - truth[c]

        vmax = max(np.nanmax(np.abs(truth[c])), np.nanmax(np.abs(pred[c])))
        evmax = np.nanmax(np.abs(err))

        if vmax == 0 or not np.isfinite(vmax):
            vmax = 1.0
        if evmax == 0 or not np.isfinite(evmax):
            evmax = 1.0

        fig, axs = plt.subplots(1, 3, figsize=(15, 4))

        im0 = axs[0].imshow(truth[c], origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        axs[0].set_title(f"FD {name}")
        plt.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)

        im1 = axs[1].imshow(pred[c], origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        axs[1].set_title(f"ML {name}")
        plt.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

        im2 = axs[2].imshow(err, origin="lower", cmap="RdBu_r", vmin=-evmax, vmax=evmax)
        axs[2].set_title(f"ML - FD {name}")
        plt.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

        for ax in axs:
            ax.set_xticks([])
            ax.set_yticks([])

        fig.suptitle(f"{ic_key} final lead={lead_index}: {name}")
        path = os.path.join(outdir, f"{ic_key}_final_{name}.png")
        plt.savefig(path, dpi=170, bbox_inches="tight")
        plt.close()


# ------------------------------------------------------------
# MAIN EVALUATION
# ------------------------------------------------------------
ic_dirs = find_ic_dirs()
print("[ICs selected]")
for d in ic_dirs:
    print(" ", os.path.basename(d))

all_rows = []
summary_rows = []

with torch.no_grad():
    for ic_dir in ic_dirs:
        ic_key = os.path.basename(ic_dir)
        print("\nEvaluating:", ic_key)

        files = sorted(
            glob.glob(os.path.join(ic_dir, "klein_step_*.npz")),
            key=step_from_file,
        )

        if len(files) < 2:
            print("  Skipping: not enough snapshots")
            continue

        if cfg.MAX_ROLLOUT_STEPS is not None:
            files = files[:cfg.MAX_ROLLOUT_STEPS + 1]

        x0_np, t0 = load_state(files[0])
        x = torch.from_numpy(x0_np[None]).to(device)

        rows = [{
            "ic_key": ic_key,
            "lead_index": 0,
            "fd_step": step_from_file(files[0]),
            "time_sec": t0,
            "eta_rmse": 0.0,
            "u_rmse": 0.0,
            "v_rmse": 0.0,
            "fd_ke": kinetic_energy(x0_np),
            "ml_ke": kinetic_energy(x0_np),
            "ke_ratio": 1.0,
        }]

        final_truth = x0_np
        final_pred = x0_np.copy()

        for lead in range(1, len(files)):
            x = model.forward_one_step(x, cfg.DT_MACRO)
            x = apply_inference_diffusion(x)

            pred = x.detach().cpu().numpy()[0]
            truth, t = load_state(files[lead])

            fd_ke = kinetic_energy(truth)
            ml_ke = kinetic_energy(pred)
            ke_ratio = ml_ke / fd_ke if fd_ke != 0 else np.nan

            row = {
                "ic_key": ic_key,
                "lead_index": lead,
                "fd_step": step_from_file(files[lead]),
                "time_sec": t,
                "eta_rmse": rmse(pred[0], truth[0]),
                "u_rmse": rmse(pred[1], truth[1]),
                "v_rmse": rmse(pred[2], truth[2]),
                "fd_ke": fd_ke,
                "ml_ke": ml_ke,
                "ke_ratio": ke_ratio,
            }
            rows.append(row)

            final_truth = truth
            final_pred = pred

            if lead % 5 == 0 or lead == len(files) - 1:
                print(
                    f"  lead={lead:03d} step={row['fd_step']:05d} "
                    f"eta_rmse={row['eta_rmse']:.4e} "
                    f"u_rmse={row['u_rmse']:.4e} "
                    f"v_rmse={row['v_rmse']:.4e} "
                    f"KE_ratio={row['ke_ratio']:.4f}"
                )

        df_ic = pd.DataFrame(rows)
        all_rows.extend(rows)

        ic_outdir = os.path.join(cfg.OUT_DIR, ic_key)
        os.makedirs(ic_outdir, exist_ok=True)

        csv_path = os.path.join(ic_outdir, f"{ic_key}_rollout_metrics.csv")
        df_ic.to_csv(csv_path, index=False)

        plot_rmse(df_ic, ic_key, ic_outdir)
        plot_ke(df_ic, ic_key, ic_outdir)
        plot_final_fields(final_truth, final_pred, ic_key, len(files) - 1, ic_outdir)

        last = df_ic.iloc[-1]
        summary_rows.append({
            "ic_key": ic_key,
            "n_leads": len(files) - 1,
            "final_step": int(last["fd_step"]),
            "final_eta_rmse": float(last["eta_rmse"]),
            "final_u_rmse": float(last["u_rmse"]),
            "final_v_rmse": float(last["v_rmse"]),
            "final_fd_ke": float(last["fd_ke"]),
            "final_ml_ke": float(last["ml_ke"]),
            "final_ke_ratio": float(last["ke_ratio"]),
            "metrics_csv": csv_path,
        })

df_all = pd.DataFrame(all_rows)
df_summary = pd.DataFrame(summary_rows)

all_csv = os.path.join(cfg.OUT_DIR, "all_rollout_metrics.csv")
summary_csv = os.path.join(cfg.OUT_DIR, "summary_eval.csv")

df_all.to_csv(all_csv, index=False)
df_summary.to_csv(summary_csv, index=False)

print("\n[DONE]")
print("All metrics:", all_csv)
print("Summary:", summary_csv)
print("\nSummary table:")
display(df_summary)


Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[AutoDetect] checking data roots...
  /content/AI_EMUL_LOCAL/klein_ckpt_1L_big50_combined
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_combined
     valid IC dirs = 68, total pairs = 1632
[AutoDetect] using DATA_DIR: /content/drive/MyDrive/AI_EMUL3/klein_ckpt_1L_big50_combined
[Load] checkpoint: /content/drive/MyDrive/AI_EMUL3/training_runs/resunet_axial_b96_big50plus_t8_arakawa_spec_lr5e5_e100/best_model.pt
[Loaded checkpoint]
checkpoint epoch: 96
model_class: MultiStepContinuousResUNetAxialModel
[ICs selected]
  gauss_00
  gauss_01
  gauss_02
  gauss_03
  gauss_04
  gauss_05
  gauss_aug_000
  gauss_aug_001
  gauss_aug_002
  gauss_aug_003
  gauss_aug_004
  gauss_aug_005
  gauss_aug_006
  gauss_aug_007
  gauss_aug_008
  gauss_aug_009
  gauss_aug_010
  gauss_aug_011
  gauss_aug_012
  gauss_aug_013
  gauss_aug_014
  gauss_aug_015
  gauss_aug_016
  gauss_aug_017
  gauss_aug_018


,ic_key,n_leads,final_step,final_eta_rmse,final_u_rmse,final_v_rmse,final_fd_ke,final_ml_ke,final_ke_ratio,metrics_csv
0,gauss_00,24,1200,1.779945,0.119135,0.122474,90.029243,104.986832,1.166141,/content/drive/MyDrive/AI_EMUL3/training_runs/...
1,gauss_01,24,1200,3.801624,0.423081,0.328999,2907.754639,2452.468018,0.843423,/content/drive/MyDrive/AI_EMUL3/training_runs/...
2,gauss_02,24,1200,2.364059,0.248677,0.252350,877.122253,828.229980,0.944258,/content/drive/MyDrive/AI_EMUL3/training_runs/...
3,gauss_03,24,1200,5.559446,0.414044,0.474759,2584.798096,2368.707031,0.916399,/content/drive/MyDrive/AI_EMUL3/training_runs/...
4,gauss_04,24,1200,5.897707,0.649369,0.496611,5031.967285,4277.342773,0.850034,/content/drive/MyDrive/AI_EMUL3/training_runs/...
5,gauss_05,24,1200,1.082313,0.102704,0.212744,68.973938,92.979401,1.348037,/content/drive/MyDrive/AI_EMUL3/training_runs/...
6,gauss_aug_000,24,1200,6.105902,0.462159,0.397694,1967.087036,1738.735352,0.883914,/content/drive/MyDrive/AI_EMUL3/training_runs/...
7,gauss_aug_001,24,1200,1.942358,0.187922,0.181780,454.905884,433.354645,0.952625,/content/drive/MyDrive/AI_EMUL3/training_runs/...
8,gauss_aug_002,24,1200,1.815429,0.117862,0.117869,82.874336,96.548256,1.164996,/content/drive/MyDrive/AI_EMUL3/training_runs/...
9,gauss_aug_003,24,1200,5.270532,0.609808,0.471785,4400.806152,3793.081543,0.861906,/content/drive/MyDrive/AI_EMUL3/training_runs/...


In [ ]:
!ls /content/drive/MyDrive/AI_EMUL3

training_runs


In [ ]:
!find /content/drive/MyDrive/AI_EMUL3 -maxdepth 2 -type d -name "klein_ckpt*"